In [1]:
# ============================================================
# CELL 1: Environment Setup and Imports
#
# PURPOSE: Confirm the environment is correctly configured
#          before any test code runs. Failures here mean
#          installation problems, not code problems.
#
# WHAT WE CHECK:
#   - Python path includes project root
#   - All required libraries are importable
#   - Config values match expectations
#   - The tokenizer module exists and is importable
# ============================================================

import sys
import os
import json
import logging
import time
import traceback
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Mapping, cast


# ── Path Setup ────────────────────────────────────────────────
# We resolve the path so we get an absolute path regardless
# of where this notebook was opened from.
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root))

print("=" * 60)
print("CELL 1: Environment Setup")
print("=" * 60)

# ── Library Imports ───────────────────────────────────────────
# Import each library separately so we get specific error
# messages when something is missing.

print("\nImporting core libraries...")

try:
    import torch
    print(f"  ✓ torch {torch.__version__}")
except ImportError as e:
    raise ImportError(
        f"PyTorch not found: {e}\n"
        f"Install with: pip install torch"
    )

try:
    import numpy as np
    print(f"  ✓ numpy {np.__version__}")
except ImportError:
    raise ImportError("NumPy not found. Install with: pip install numpy")

try:
    from transformers import AutoTokenizer
    import transformers
    print(f"  ✓ transformers {transformers.__version__}")
except ImportError:
    raise ImportError(
        "transformers not found. Install with: pip install transformers"
    )

# ── Project Imports ───────────────────────────────────────────
print("\nImporting project modules...")

try:
    import config
    print(f"  ✓ config module loaded")
except ImportError as e:
    raise ImportError(
        f"Cannot import config: {e}\n"
        f"Project root: {project_root}\n"
        f"Check that config.py exists at {project_root}/config.py"
    )

try:
    from modules.tokenizer import NanoJEPATokenizer
    print(f"  ✓ NanoJEPATokenizer loaded")
except ImportError as e:
    raise ImportError(
        f"Cannot import NanoJEPATokenizer: {e}\n"
        f"Check that modules/tokenizer.py exists"
    )

# ── Config Validation ─────────────────────────────────────────
print("\nValidating config values...")

required_config_attrs = [
    "VOCAB_SIZE",
    "MAX_SEQ_LEN",
    "VOCAB_FILE",
]

for attr in required_config_attrs:
    if not hasattr(config, attr):
        raise AttributeError(
            f"config.py is missing required attribute: {attr}"
        )
    val = getattr(config, attr)
    print(f"  ✓ config.{attr} = {val}")

# Validate specific values we depend on
assert config.VOCAB_SIZE == 32000, (
    f"Expected VOCAB_SIZE=32000, got {config.VOCAB_SIZE}. "
    f"This affects embedding table size in Module 2."
)

assert config.MAX_SEQ_LEN == 256, (
    f"Expected MAX_SEQ_LEN=256, got {config.MAX_SEQ_LEN}. "
    f"This affects positional encoding in Module 2."
)

assert config.MAX_SEQ_LEN > 0, "MAX_SEQ_LEN must be positive"
assert config.VOCAB_SIZE > 3, "VOCAB_SIZE must be > 3 (need room for PAD/BOS/EOS)"

# ── Device Info ───────────────────────────────────────────────
print("\nHardware info:")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Store globals for later cells
CELL1_RESULTS = {
    "project_root": str(project_root),
    "torch_version": torch.__version__,
    "device": device,
    "vocab_size": config.VOCAB_SIZE,
    "max_seq_len": config.MAX_SEQ_LEN,
}

print(f"\nProject root: {project_root}")
print("\n" + "=" * 60)
print("✅ CELL 1 PASSED: Environment correctly configured")
print("=" * 60)

CELL 1: Environment Setup

Importing core libraries...
  ✓ torch 2.12.0+cpu
  ✓ numpy 2.4.6


c:\Users\harsh\Desktop\PRO\JEPA - 2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✓ transformers 5.10.2

Importing project modules...
  ✓ config module loaded
  ✓ NanoJEPATokenizer loaded

Validating config values...
  ✓ config.VOCAB_SIZE = 32000
  ✓ config.MAX_SEQ_LEN = 256
  ✓ config.VOCAB_FILE = C:\Users\harsh\Desktop\PRO\JEPA - 2\data\vocab_map.json

Hardware info:
  Device: cpu

Project root: C:\Users\harsh\Desktop\PRO\JEPA - 2

✅ CELL 1 PASSED: Environment correctly configured


In [2]:
# ============================================================
# CELL 2: Load Dataset for Vocabulary Building
#
# PURPOSE: Get real math text to build a vocabulary that
#          actually covers GSM8K problems.
#
# WHY THIS MATTERS:
#   If we build vocab from random text, we might not have
#   tokens for "quotient", "remainder", "hypotenuse", etc.
#   The vocabulary coverage test in Cell 9 catches this.
#
# FALLBACK: If GSM8K isn't available (offline, no HF token),
#   we use a synthetic corpus that covers math vocabulary.
#   This is sufficient for structural testing but Cell 9
#   will skip the coverage assertion.
# ============================================================

print("=" * 60)
print("CELL 2: Load Dataset for Vocabulary Building")
print("=" * 60)

# ── Attempt to Load Real GSM8K Data ───────────────────────────
USING_REAL_DATA = False
all_texts: List[str] = []

try:
    from datasets import load_dataset
    print("\nAttempting to load GSM8K from HuggingFace...")
    print("(This downloads ~10MB on first run, then uses cache)")

    dataset = load_dataset("gsm8k", "main", trust_remote_code=True)

    # Collect questions and answers from train split
    # We use both because answers contain different vocabulary
    # (step-by-step reasoning, "####" answer markers, etc.)
    for example in dataset["train"]:
        example = cast(Mapping[str, Any], example)
        question = example["question"].strip()
        answer = example["answer"].strip()
        if question:
            all_texts.append(question)
        if answer:
            all_texts.append(answer)

    USING_REAL_DATA = True
    print(f"  ✓ Loaded {len(all_texts):,} texts from GSM8K train split")
    print(f"  ✓ Train examples: {len(dataset['train']):,}")
    print(f"  ✓ Test examples:  {len(dataset['test']):,}")

    # Show what we're working with
    sample_q = dataset["train"][0]["question"]
    sample_a = dataset["train"][0]["answer"]
    print(f"\nSample question: {sample_q[:80]}...")
    print(f"Sample answer:   {sample_a[:80]}...")

except Exception as e:
    print(f"\nGSM8K load failed: {type(e).__name__}: {e}")
    print("Falling back to synthetic math corpus...")

    # This corpus is carefully designed to cover:
    # - Algebra problems (variables, equations)
    # - Arithmetic (fractions, decimals, percentages)
    # - Word problems (real-world contexts)
    # - Multi-step reasoning (Step 1, Step 2, etc.)
    # - Answer formats ($, %, units)
    synthetic_base = [
        # Algebra
        "If 3x + 9 = 21, find x",
        "Solve for y: 2y - 7 = 13",
        "What is the value of z if 4z + 16 = 48?",
        "Simplify: (x + 3)(x - 2)",
        "Factor the expression: x^2 + 5x + 6",
        # Arithmetic word problems
        "John has 5 apples and buys 3 more. How many does he have?",
        "Maria spent $24.50 on groceries and $8.75 on coffee. Total?",
        "A store has 150 items. It sells 42. How many remain?",
        # Rates and percentages
        "A train travels at 60 mph for 2.5 hours. What is the distance?",
        "Calculate 15% of $240",
        "If a shirt costs $45 and is on sale for 20% off, what is the price?",
        "The price increased from $80 to $96. What percent increase?",
        # Multi-step solutions
        "Step 1: Subtract 9 from both sides. 3x = 12",
        "Step 2: Divide both sides by 3. x = 4",
        "Therefore, x = 4. The answer is 4.",
        "First, multiply both sides by 2. Then, subtract 6.",
        # Answer formats
        "The answer is $4.25",
        "Total cost = 4 × $0.50 + 3 × $0.75 = $2.00 + $2.25 = $4.25",
        "She earns $12.50 per hour and works 8 hours. Total: $100.00",
        # Fractions and ratios
        "1/3 of 90 is 30",
        "The ratio of boys to girls is 3:2",
        "Two-fifths of the students passed. How many failed?",
        # Geometry basics
        "Area = length × width = 12 × 8 = 96 square meters",
        "Perimeter = 2(l + w) = 2(12 + 8) = 40 meters",
        # Statistics
        "The average of 5, 10, 15 is (5 + 10 + 15) / 3 = 10",
        "Find the median of: 3, 7, 9, 12, 15",
    ]

    # Repeat to simulate a larger corpus (vocab building needs repetition)
    all_texts = synthetic_base * 200
    dataset = None  # No real dataset available

    print(f"  ✓ Synthetic corpus: {len(all_texts):,} texts")
    print(f"  ✓ Unique base texts: {len(synthetic_base)}")

# ── Corpus Statistics ─────────────────────────────────────────
print("\nCorpus statistics:")

# Text length distribution
lengths = [len(t.split()) for t in all_texts[:1000]]  # sample
avg_len = sum(lengths) / len(lengths)
max_len = max(lengths)
min_len = min(lengths)

print(f"  Total texts:     {len(all_texts):,}")
print(f"  Avg word count:  {avg_len:.1f}")
print(f"  Max word count:  {max_len}")
print(f"  Min word count:  {min_len}")

# Unique vocabulary (word-level, just for inspection)
word_set = set()
for text in all_texts[:500]:  # sample
    word_set.update(text.lower().split())
print(f"  Unique words (sample): {len(word_set):,}")

# Store for later cells
CELL2_RESULTS = {
    "using_real_data": USING_REAL_DATA,
    "n_texts": len(all_texts),
    "dataset": dataset if USING_REAL_DATA else None,
}

print("\n" + "=" * 60)
print("✅ CELL 2 PASSED: Data loaded successfully")
print(f"   Using real GSM8K data: {USING_REAL_DATA}")
print("=" * 60)

CELL 2: Load Dataset for Vocabulary Building


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'gsm8k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
2026-06-12 15:37:32,778 | datasets.load | ERROR | `trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'gsm8k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



Attempting to load GSM8K from HuggingFace...
(This downloads ~10MB on first run, then uses cache)


2026-06-12 15:37:33,502 | httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-12 15:37:33,786 | httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-12 15:37:33,842 | httpx | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-12 15:37:34,133 | httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 307 Temporary Redirect"
2026-06-12 15:37:34,421 | httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-12 15:37:34,422 | huggingface_hub.utils._http | WARNING | Warning: You are sending unauthenticated reques


GSM8K load failed: HfUriError: Invalid HF URI 'hf://datasets/gsm8k@740312add88f781978c0658806c59bc2815b9866/.huggingface.yaml'. Repository id must be 'namespace/name', got 'gsm8k'.
Falling back to synthetic math corpus...
  ✓ Synthetic corpus: 5,200 texts
  ✓ Unique base texts: 26

Corpus statistics:
  Total texts:     5,200
  Avg word count:  9.9
  Max word count:  16
  Min word count:  4
  Unique words (sample): 162

✅ CELL 2 PASSED: Data loaded successfully
   Using real GSM8K data: False


In [3]:
# ============================================================
# CELL 3: Build Vocabulary and Inspect Internal State
# ============================================================

print("=" * 60)
print("CELL 3: Build Vocabulary and Inspect Internal State")
print("=" * 60)

# Single source of truth for the reserved-slot layout.
# Must match NanoJEPATokenizer.__init__ and build_vocab(start=5).
RESERVED_SLOTS = {
    0: "PAD",
    1: "BOS",
    2: "EOS",
    3: "unused",
    4: "UNK",
}
FIRST_LEARNED_ID = max(RESERVED_SLOTS.keys()) + 1   # = 5

# ── Initialize Tokenizer ──────────────────────────────────────
print("\nInitializing NanoJEPATokenizer...")
tok = NanoJEPATokenizer()

print("\nBuilding/loading vocabulary...")
print(
    "(First run: builds vocab, ~10 seconds. "
    "Subsequent: loads from disk, ~1 second)"
)

build_start   = time.perf_counter()
tok.build_vocab(all_texts)
build_elapsed = time.perf_counter() - build_start

print(f"Vocab build/load time: {build_elapsed:.2f}s")

# ── Special Token Verification ────────────────────────────────
print("\n── Special Token IDs ────────────────────────────────────")

special_tokens = {
    "PAD": (tok.pad_id, 0),
    "BOS": (tok.bos_id, 1),
    "EOS": (tok.eos_id, 2),
    "UNK": (tok.unk_id, 4),
}

for name, (actual, expected) in special_tokens.items():
    status = "✓" if actual == expected else "✗"
    print(f"  {status} {name}: {actual} (expected {expected})")
    assert actual == expected, (
        f"Special token {name} has wrong ID!\n"
        f"  Expected: {expected}\n"
        f"  Got:      {actual}\n"
        f"  This will corrupt every sequence ever encoded."
    )

# ── Mapping Structure ─────────────────────────────────────────
print("\n── Vocabulary Mapping Structure ─────────────────────────")

n_mapped = len(tok.bloom_to_reduced)
print(f"  BLOOM tokens mapped:  {n_mapped:,}")
print(f"  Target vocab size:    {config.VOCAB_SIZE:,}")
print(
    f"  Reserved slots:       "
    f"{', '.join(f'{v}={k}' for k, v in RESERVED_SLOTS.items())}"
)
print(f"  Learned tokens start: ID {FIRST_LEARNED_ID}")

assert n_mapped > 0, (
    "Vocabulary is empty! build_vocab() failed silently."
)
assert n_mapped <= config.VOCAB_SIZE, (
    f"Mapped {n_mapped} tokens but VOCAB_SIZE={config.VOCAB_SIZE}. "
    f"Reduced IDs would overflow!"
)

# ── Injectivity Check ─────────────────────────────────────────
print("\n── Injectivity Check (no collisions) ───────────────────")

seen: dict[int, list[int]] = {}   # reduced_id → [bloom_ids...]
for bloom_id, reduced_id in tok.bloom_to_reduced.items():
    seen.setdefault(reduced_id, []).append(bloom_id)

collisions = [
    (rid, bids) for rid, bids in seen.items() if len(bids) > 1
]

print(f"  Unique reduced IDs used: {len(seen):,}")
print(f"  Collisions found:        {len(collisions)}")

if collisions:
    print(f"\n  ⚠ COLLISIONS DETECTED (first 5):")
    for reduced_id, bloom_ids in collisions[:5]:
        print(f"    reduced_id={reduced_id} ← bloom_ids={bloom_ids[:3]}...")
    print(
        f"\n  Note: Each collision means two BLOOM tokens look identical "
        f"to the model.\n"
        f"  Verify this is intentional in your tokenizer design."
    )
else:
    print(f"  ✓ No collisions — mapping is injective")

# ── Reduced ID Range Check ────────────────────────────────────
print("\n── Reduced ID Range Check ───────────────────────────────")

all_reduced_ids = list(tok.bloom_to_reduced.values())
min_reduced     = min(all_reduced_ids)
max_reduced     = max(all_reduced_ids)

print(f"  Reduced ID range: [{min_reduced}, {max_reduced}]")
print(f"  Valid range:      [{FIRST_LEARNED_ID}, {config.VOCAB_SIZE - 1}]")
print(
    f"  Reserved slots:   "
    f"{', '.join(f'{v}={k}' for k, v in RESERVED_SLOTS.items())}"
)

# All reserved IDs (0-4) must be absent from learned mappings
invalid_ids = [rid for rid in all_reduced_ids if rid < FIRST_LEARNED_ID]
if invalid_ids:
    overlap_names = {
        rid: RESERVED_SLOTS.get(rid, "reserved")
        for rid in sorted(set(invalid_ids))
    }
    print(f"\n  ✗ {len(invalid_ids)} reduced IDs overlap with reserved slots!")
    for rid, name in overlap_names.items():
        print(f"    ID {rid} ({name}) used by a learned token")
    raise AssertionError(
        f"Reduced IDs must not overlap with reserved slots 0–{FIRST_LEARNED_ID - 1}.\n"
        f"  Slot layout: {RESERVED_SLOTS}\n"
        f"  Learned tokens must start at {FIRST_LEARNED_ID} "
        f"(build_vocab uses enumerate(..., start={FIRST_LEARNED_ID})).\n"
        f"  Found {len(invalid_ids)} violations: "
        f"{sorted(set(invalid_ids))}"
    )
else:
    print(f"  ✓ No overlap with reserved slots (0–{FIRST_LEARNED_ID - 1})")

# Upper-bound check: no ID may reach or exceed VOCAB_SIZE
if max_reduced >= config.VOCAB_SIZE:
    raise AssertionError(
        f"Max reduced ID ({max_reduced}) >= VOCAB_SIZE ({config.VOCAB_SIZE}).\n"
        f"  Embedding lookup will throw an index-out-of-range error at runtime."
    )
print(f"  ✓ All IDs within valid range [{FIRST_LEARNED_ID}, {config.VOCAB_SIZE - 1}]")

# ── Forced-Token Verification ─────────────────────────────────
# Confirm that every symbol in FORCED_VOCAB_TOKENS survived
# the top-K cut and is actually present in the vocab.
print("\n── Forced-Token Coverage Check ──────────────────────────")

missing_symbols: list[str] = []
for symbol in NanoJEPATokenizer.FORCED_VOCAB_TOKENS:
    bloom_ids_for_symbol = tok._bloom_tok.encode(
        symbol, add_special_tokens=False
    )
    for bid in bloom_ids_for_symbol:
        if bid not in tok.bloom_to_reduced:
            missing_symbols.append(symbol)
            break   # one missing bloom ID is enough to flag the symbol

if missing_symbols:
    print(f"  ✗ {len(missing_symbols)} forced symbol(s) NOT in vocab:")
    for s in missing_symbols:
        print(f"    '{s}'")
    raise AssertionError(
        f"Forced tokens missing from vocab: {missing_symbols}\n"
        f"  Delete {config.VOCAB_FILE} and rebuild."
    )
else:
    print(
        f"  ✓ All {len(NanoJEPATokenizer.FORCED_VOCAB_TOKENS)} forced symbols "
        f"are present in vocab"
    )

# ── Vocab File Persistence ────────────────────────────────────
print("\n── Vocab File Persistence ───────────────────────────────")

vocab_path = Path(config.VOCAB_FILE)
print(f"  Vocab file path: {vocab_path}")
print(f"  File exists:     {vocab_path.exists()}")

if not vocab_path.exists():
    raise FileNotFoundError(
        f"Vocab file not created at {vocab_path}. "
        f"build_vocab() should write this file."
    )

with open(vocab_path, "r") as f:
    vocab_data = json.load(f)

print(f"  File size:       {vocab_path.stat().st_size / 1024:.1f} KB")
print(f"  JSON keys:       {list(vocab_data.keys())}")

required_keys = {"bloom_to_reduced", "reduced_to_bloom"}
missing_keys  = required_keys - set(vocab_data.keys())
assert not missing_keys, (
    f"Vocab JSON missing keys: {missing_keys}\n"
    f"  Found: {list(vocab_data.keys())}"
)
print(f"  ✓ All required JSON keys present")

# ── Fresh Load Test ───────────────────────────────────────────
print("\n── Fresh Load Test (simulates inference deployment) ─────")

tok_fresh  = NanoJEPATokenizer()
load_start = time.perf_counter()
tok_fresh.build_vocab([])          # empty list → must load from disk
load_elapsed = time.perf_counter() - load_start

print(f"  Fresh load time: {load_elapsed:.3f}s")
print(f"  Tokens loaded:   {len(tok_fresh.bloom_to_reduced):,}")

assert len(tok_fresh.bloom_to_reduced) == len(tok.bloom_to_reduced), (
    f"Fresh load has different vocab size!\n"
    f"  Original: {len(tok.bloom_to_reduced):,}\n"
    f"  Reloaded: {len(tok_fresh.bloom_to_reduced):,}\n"
    f"  Inference would use different token IDs than training."
)

sample_bloom_ids = list(tok.bloom_to_reduced.keys())[:10]
for bloom_id in sample_bloom_ids:
    orig  = tok.bloom_to_reduced[bloom_id]
    fresh = tok_fresh.bloom_to_reduced[bloom_id]
    assert orig == fresh, (
        f"Bloom ID {bloom_id} maps differently after reload!\n"
        f"  Original: {orig}\n"
        f"  Reloaded: {fresh}"
    )
print(f"  ✓ Spot-checked 10 mappings — consistent after reload")

# ── Store Results ─────────────────────────────────────────────
CELL3_RESULTS = {
    "tok":              tok,
    "n_mapped":         n_mapped,
    "build_time_sec":   build_elapsed,
    "load_time_sec":    load_elapsed,
    "n_collisions":     len(collisions),
    "reduced_id_range": (min_reduced, max_reduced),
}

print("\n" + "=" * 60)
print("✅ CELL 3 PASSED: Vocabulary built and validated")
print(
    f"   Tokens: {n_mapped:,} | "
    f"Build: {build_elapsed:.2f}s | "
    f"Load: {load_elapsed:.3f}s"
)
print("=" * 60)

CELL 3: Build Vocabulary and Inspect Internal State

Initializing NanoJEPATokenizer...


2026-06-12 15:37:38,478 | httpx | INFO | HTTP Request: HEAD https://huggingface.co/bigscience/bloom-560m/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-12 15:37:38,535 | httpx | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/bigscience/bloom-560m/ac2ae5fab2ce3f9f40dc79b5ca9f637430d24971/config.json "HTTP/1.1 200 OK"
2026-06-12 15:37:38,834 | httpx | INFO | HTTP Request: HEAD https://huggingface.co/bigscience/bloom-560m/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-12 15:37:38,892 | httpx | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/bigscience/bloom-560m/ac2ae5fab2ce3f9f40dc79b5ca9f637430d24971/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-12 15:37:39,185 | httpx | INFO | HTTP Request: GET https://huggingface.co/api/models/bigscience/bloom-560m/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-06-12 15:37:39,476 | httpx | INFO | HTTP


Building/loading vocabulary...
(First run: builds vocab, ~10 seconds. Subsequent: loads from disk, ~1 second)


2026-06-12 15:37:42,620 | modules.tokenizer | INFO |   Tokenizing text 3,000/5,200...
2026-06-12 15:37:42,708 | modules.tokenizer | INFO |   Tokenizing text 4,000/5,200...
2026-06-12 15:37:42,787 | modules.tokenizer | INFO |   Tokenizing text 5,000/5,200...
2026-06-12 15:37:42,813 | modules.tokenizer | INFO |   Force-including 192 bloom IDs for 209 critical symbols.
2026-06-12 15:37:42,814 | modules.tokenizer | INFO |   ✓ All 192 forced symbol IDs are in vocab.
2026-06-12 15:37:42,819 | modules.tokenizer | INFO | Vocab built: 347 tokens saved to C:\Users\harsh\Desktop\PRO\JEPA - 2\data\vocab_map.json


Vocab build/load time: 0.45s

── Special Token IDs ────────────────────────────────────
  ✓ PAD: 0 (expected 0)
  ✓ BOS: 1 (expected 1)
  ✓ EOS: 2 (expected 2)
  ✓ UNK: 4 (expected 4)

── Vocabulary Mapping Structure ─────────────────────────
  BLOOM tokens mapped:  347
  Target vocab size:    32,000
  Reserved slots:       PAD=0, BOS=1, EOS=2, unused=3, UNK=4
  Learned tokens start: ID 5

── Injectivity Check (no collisions) ───────────────────
  Unique reduced IDs used: 347
  Collisions found:        0
  ✓ No collisions — mapping is injective

── Reduced ID Range Check ───────────────────────────────
  Reduced ID range: [5, 351]
  Valid range:      [5, 31999]
  Reserved slots:   PAD=0, BOS=1, EOS=2, unused=3, UNK=4
  ✓ No overlap with reserved slots (0–4)
  ✓ All IDs within valid range [5, 31999]

── Forced-Token Coverage Check ──────────────────────────
  ✓ All 209 forced symbols are present in vocab

── Vocab File Persistence ───────────────────────────────
  Vocab file path: C:\Us

2026-06-12 15:37:43,160 | httpx | INFO | HTTP Request: HEAD https://huggingface.co/bigscience/bloom-560m/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-12 15:37:43,219 | httpx | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/bigscience/bloom-560m/ac2ae5fab2ce3f9f40dc79b5ca9f637430d24971/config.json "HTTP/1.1 200 OK"
2026-06-12 15:37:43,548 | httpx | INFO | HTTP Request: HEAD https://huggingface.co/bigscience/bloom-560m/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-12 15:37:43,608 | httpx | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/bigscience/bloom-560m/ac2ae5fab2ce3f9f40dc79b5ca9f637430d24971/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-12 15:37:43,906 | httpx | INFO | HTTP Request: GET https://huggingface.co/api/models/bigscience/bloom-560m/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-06-12 15:37:44,201 | httpx | INFO | HTTP

  Fresh load time: 0.003s
  Tokens loaded:   347
  ✓ Spot-checked 10 mappings — consistent after reload

✅ CELL 3 PASSED: Vocabulary built and validated
   Tokens: 347 | Build: 0.45s | Load: 0.003s


In [4]:
# ============================================================
# CELL 4: Core Encoding Test — Step-by-Step Trace
#
# PURPOSE: Follow "If 3x + 9 = 21, find x" through the
#          complete encoding pipeline and verify every
#          structural requirement.
#
# WHY THIS SPECIFIC STRING:
#   - Has variables (x)          → tests symbol encoding
#   - Has operators (+, =)       → tests ASCII operator handling
#   - Has a number (3, 9, 21)    → tests numeric encoding
#   - Has punctuation (,)        → tests punctuation handling
#   - Is short enough            → leaves room to verify padding
#   - Is a realistic math query  → representative of training data
#
# REQUIREMENTS WE VERIFY:
#   R4.1: output has 'input_ids' and 'attention_mask' keys
#   R4.2: both tensors have dtype=torch.long (int64)
#   R4.3: both tensors have shape (MAX_SEQ_LEN,) = (256,)
#   R4.4: position 0 is BOS token
#   R4.5: last real token is EOS token
#   R4.6: all positions after EOS are PAD (value=0)
#   R4.7: attention_mask is 1 for real tokens, 0 for padding
#   R4.8: all token IDs are in range [0, VOCAB_SIZE)
# ============================================================

print("=" * 60)
print("CELL 4: Core Encoding Test — Step-by-Step Trace")
print("=" * 60)

EXAMPLE = "If 3x + 9 = 21, find x"
print(f"\nInput string: '{EXAMPLE}'")
print(f"Length: {len(EXAMPLE)} characters, {len(EXAMPLE.split())} words")

# ── Run Encoding ─────────────────────────────────────────────
print("\nRunning tok.encode()...")
result = tok.encode(EXAMPLE)

# ── R4.1: Required Keys ───────────────────────────────────────
print("\n── R4.1: Required Output Keys ───────────────────────────")
required_keys = ["input_ids", "attention_mask"]
for key in required_keys:
    present = key in result
    print(f"  {'✓' if present else '✗'} '{key}' in output")
    assert present, f"Missing required key: '{key}'"

extra_keys = set(result.keys()) - set(required_keys)
if extra_keys:
    print(f"  ℹ Extra keys (acceptable): {extra_keys}")

# ── R4.2: Data Types ──────────────────────────────────────────
print("\n── R4.2: Tensor Data Types ───────────────────────────────")
for key in required_keys:
    tensor = result[key]
    dtype = tensor.dtype
    correct = dtype == torch.long
    print(f"  {'✓' if correct else '✗'} {key}.dtype = {dtype} (expected torch.long/int64)")
    assert correct, (
        f"result['{key}'].dtype = {dtype}, expected torch.long\n"
        f"Module 2 (embeddings) calls nn.Embedding()(input_ids) which "
        f"requires LongTensor. Wrong dtype will crash at embedding lookup."
    )

# ── R4.3: Tensor Shapes ───────────────────────────────────────
print("\n── R4.3: Tensor Shapes ──────────────────────────────────")
expected_shape = (config.MAX_SEQ_LEN,)
for key in required_keys:
    shape = result[key].shape
    correct = shape == expected_shape
    print(f"  {'✓' if correct else '✗'} {key}.shape = {shape} (expected {expected_shape})")
    assert correct, (
        f"result['{key}'].shape = {shape}, expected {expected_shape}\n"
        f"Fixed shape is required for batching in torch.stack()."
    )

# ── Compute Token Boundaries ──────────────────────────────────
mask = result["attention_mask"]
n_real_tokens = mask.sum().item()
n_padding = config.MAX_SEQ_LEN - n_real_tokens
real_ids = result["input_ids"][:n_real_tokens].tolist()

print(f"\n── Token Count Analysis ─────────────────────────────────")
print(f"  Total positions:  {config.MAX_SEQ_LEN}")
print(f"  Real tokens:      {n_real_tokens}  (includes BOS and EOS)")
print(f"  Padding tokens:   {n_padding}")
print(f"  Real token IDs:   {real_ids}")

# Sanity: short input should produce fewer than MAX_SEQ_LEN real tokens
assert n_real_tokens < config.MAX_SEQ_LEN, (
    f"Short input '{EXAMPLE}' used all {config.MAX_SEQ_LEN} positions. "
    f"Either padding isn't being added or truncation fired incorrectly."
)
assert n_real_tokens >= 3, (
    f"Too few real tokens: {n_real_tokens}. "
    f"Minimum is 3: BOS + at least 1 content token + EOS."
)

# ── R4.4: BOS Token ───────────────────────────────────────────
print(f"\n── R4.4: BOS Token Position ─────────────────────────────")
bos_actual = real_ids[0]
print(f"  Position 0: {bos_actual} (expected BOS={tok.bos_id})")
assert bos_actual == tok.bos_id, (
    f"First token must be BOS.\n"
    f"  Expected: {tok.bos_id}\n"
    f"  Got:      {bos_actual}\n"
    f"  The model uses BOS to know where a sequence starts."
)
print(f"  ✓ Position 0 is BOS")

# ── R4.5: EOS Token ───────────────────────────────────────────
print(f"\n── R4.5: EOS Token Position ─────────────────────────────")
eos_actual = real_ids[-1]
print(f"  Position {n_real_tokens-1}: {eos_actual} (expected EOS={tok.eos_id})")
assert eos_actual == tok.eos_id, (
    f"Last real token must be EOS.\n"
    f"  Expected: {tok.eos_id}\n"
    f"  Got:      {eos_actual}\n"
    f"  The model uses EOS to know where a sequence ends."
)
print(f"  ✓ Position {n_real_tokens-1} is EOS")

# ── R4.6: Padding Values ──────────────────────────────────────
print(f"\n── R4.6: Padding Token Values ───────────────────────────")
if n_padding > 0:
    padding_ids = result["input_ids"][n_real_tokens:].tolist()
    non_zero_padding = [x for x in padding_ids if x != tok.pad_id]
    print(f"  Padding region: positions [{n_real_tokens}, {config.MAX_SEQ_LEN - 1}]")
    print(f"  Non-PAD values in padding: {len(non_zero_padding)}")
    print(f"  First 5 padding values: {padding_ids[:5]}")
    assert len(non_zero_padding) == 0, (
        f"Found non-PAD tokens in padding region!\n"
        f"  Non-zero positions: {[i + n_real_tokens for i, x in enumerate(padding_ids) if x != 0]}\n"
        f"  Values: {non_zero_padding[:5]}\n"
        f"  Padding must be exactly 0 (PAD token ID)."
    )
    print(f"  ✓ All {n_padding} padding positions are PAD (value=0)")
else:
    print(f"  No padding (sequence fills entire MAX_SEQ_LEN)")

# ── R4.7: Attention Mask Values ───────────────────────────────
print(f"\n── R4.7: Attention Mask Values ──────────────────────────")
real_mask = mask[:n_real_tokens].tolist()
pad_mask = mask[n_real_tokens:].tolist()

all_ones_in_real = all(v == 1 for v in real_mask)
all_zeros_in_pad = all(v == 0 for v in pad_mask)

print(f"  Real region mask (first 5): {real_mask[:5]}")
print(f"  Pad region mask (last 5):   {pad_mask[-5:] if pad_mask else 'N/A'}")
print(f"  All real positions = 1:     {all_ones_in_real}")
print(f"  All pad positions  = 0:     {all_zeros_in_pad}")

assert all_ones_in_real, (
    f"Attention mask should be 1 for all real tokens.\n"
    f"Positions with mask=0 in real region: "
    f"{[i for i, v in enumerate(real_mask) if v != 1]}"
)
assert all_zeros_in_pad, (
    f"Attention mask should be 0 for all padding.\n"
    f"Positions with mask=1 in pad region: "
    f"{[i + n_real_tokens for i, v in enumerate(pad_mask) if v != 0]}"
)
print(f"  ✓ Attention mask is correct")

# ── R4.8: Token ID Range ──────────────────────────────────────
print(f"\n── R4.8: Token ID Range ─────────────────────────────────")
all_ids = result["input_ids"].tolist()
min_id = min(all_ids)
max_id = max(all_ids)
out_of_range = [x for x in all_ids if x < 0 or x >= config.VOCAB_SIZE]

print(f"  ID range in output: [{min_id}, {max_id}]")
print(f"  Valid range:        [0, {config.VOCAB_SIZE - 1}]")
print(f"  Out-of-range IDs:   {len(out_of_range)}")

assert len(out_of_range) == 0, (
    f"Found {len(out_of_range)} token IDs outside valid range!\n"
    f"  Out-of-range values: {out_of_range[:5]}\n"
    f"  This will cause IndexError in nn.Embedding lookup."
)
print(f"  ✓ All IDs in valid range")

# ── Visual Summary ────────────────────────────────────────────
print(f"\n── Visual Token Map ─────────────────────────────────────")
print(f"  Position | ID    | Type")
print(f"  ---------|-------|----------")
print(f"  0        | {real_ids[0]:<5} | BOS")
for i, tok_id in enumerate(real_ids[1:-1], 1):
    print(f"  {i:<8} | {tok_id:<5} | content")
print(f"  {n_real_tokens-1:<8} | {real_ids[-1]:<5} | EOS")
if n_padding > 0:
    print(f"  {n_real_tokens:<8} | 0     | PAD  ×{n_padding}")

# Store results
CELL4_RESULTS = {
    "example": EXAMPLE,
    "result": result,
    "n_real_tokens": n_real_tokens,
    "n_padding": n_padding,
    "real_ids": real_ids,
}

print("\n" + "=" * 60)
print("✅ CELL 4 PASSED: Core encoding correct")
print(f"   '{EXAMPLE}' → {n_real_tokens} real tokens + {n_padding} padding")
print("=" * 60)

CELL 4: Core Encoding Test — Step-by-Step Trace

Input string: 'If 3x + 9 = 21, find x'
Length: 22 characters, 8 words

Running tok.encode()...

── R4.1: Required Output Keys ───────────────────────────
  ✓ 'input_ids' in output
  ✓ 'attention_mask' in output

── R4.2: Tensor Data Types ───────────────────────────────
  ✓ input_ids.dtype = torch.int64 (expected torch.long/int64)
  ✓ attention_mask.dtype = torch.int64 (expected torch.long/int64)

── R4.3: Tensor Shapes ──────────────────────────────────
  ✓ input_ids.shape = torch.Size([256]) (expected (256,))
  ✓ attention_mask.shape = torch.Size([256]) (expected (256,))

── Token Count Analysis ─────────────────────────────────
  Total positions:  256
  Real tokens:      11  (includes BOS and EOS)
  Padding tokens:   245
  Real token IDs:   [1, 223, 224, 198, 209, 197, 245, 23, 246, 210, 2]

── R4.4: BOS Token Position ─────────────────────────────
  Position 0: 1 (expected BOS=1)
  ✓ Position 0 is BOS

── R4.5: EOS Token Position ───

In [5]:
# ============================================================
# CELL 5: Decode Round-Trip Test
# ============================================================

# ── Imports needed by this cell ───────────────────────────────
# These are also available from prior cells, but listed here
# explicitly so the cell is self-contained and the assert
# message f-strings don't raise NameError.
import torch
from config import VOCAB_FILE, MAX_SEQ_LEN

print("=" * 60)
print("CELL 5: Decode Round-Trip Test")
print("=" * 60)

result  = CELL4_RESULTS["result"]
EXAMPLE = CELL4_RESULTS["example"]   # "If 3x + 9 = 21, find x"

# ── Basic Decode ──────────────────────────────────────────────
print(f"\nOriginal: '{EXAMPLE}'")
print("Running tok.decode()...")

decoded = tok.decode(result["input_ids"])

print(f"Decoded:  '{decoded}'")
print(f"Type:     {type(decoded)}")

assert isinstance(decoded, str), (
    f"decode() must return str, got {type(decoded)}"
)
assert len(decoded.strip()) > 0, (
    "decode() returned empty string for a non-trivial input.\n"
    "Check that bloom_to_reduced and reduced_to_bloom are populated."
)

# ── Special Token Leakage Check ───────────────────────────────
print("\n── Special Token Leakage Check ──────────────────────────")

special_token_strings = [
    "</s>", "<s>", "<pad>",
    "[BOS]", "[EOS]", "[PAD]",
    "<|endoftext|>",
]
found_special = [s for s in special_token_strings if s in decoded]

if found_special:
    print(f"  ⚠ Found special token strings in decoded text: {found_special}")
else:
    print(f"  ✓ No special token strings leaked into decoded text")

# ── Content Preservation Check ────────────────────────────────
print("\n── Content Preservation Check ───────────────────────────")

key_numbers = ["3", "9", "21"]
key_words   = ["find", "x"]
preserved   = [e for e in key_numbers + key_words
               if e.lower() in decoded.lower()]
missing     = [e for e in key_numbers + key_words
               if e.lower() not in decoded.lower()]

print(f"  Key elements preserved: {preserved}")
print(f"  Key elements missing:   {missing}")

original_words = set(EXAMPLE.lower().replace(",", "").split())
decoded_words  = set(decoded.lower().replace(",", "").split())
overlap_count  = len(original_words & decoded_words)
overlap_ratio  = overlap_count / len(original_words)

print(f"\n  Word-level overlap:")
print(f"    Original words: {sorted(original_words)}")
print(f"    Decoded words:  {sorted(decoded_words)}")
print(f"    Overlap: {overlap_count}/{len(original_words)} = {overlap_ratio:.1%}")

assert overlap_ratio > 0.7, (
    f"Round-trip decode lost too much content!\n"
    f"  Overlap: {overlap_ratio:.1%} (threshold: 70%)\n"
    f"  Missing elements: {missing}\n"
    f"  Likely cause: vocab file is stale (pre-dates FORCED_VOCAB_TOKENS).\n"
    f"  Fix: run Cell 5-PRE to delete {VOCAB_FILE}, then rebuild."
)
print(f"  ✓ {overlap_ratio:.1%} word overlap (threshold: 70%)")

# ── Multi-Text Round-Trip (with UNK diagnostics) ──────────────
print("\n── Multi-Text Round-Trip Test ───────────────────────────")

round_trip_texts = [
    "If 3x + 9 = 21, find x",
    "Step 1: Subtract 9 from both sides",
    "The answer is 4",
    "15% of $240 = $36",
    "x = 4",
    "Total = $124.50",
]

print(f"  {'Original':<35} | {'Decoded':<35} | {'Overlap':>7}")
print(f"  {'-'*35} | {'-'*35} | {'-'*7}")

all_passed: bool      = True
failures: list[dict]  = []

for text in round_trip_texts:

    bloom_ids = tok._bloom_tok.encode(text, add_special_tokens=False)
    remapped  = tok._remap_ids(bloom_ids, log_unknowns=True)

    unk_count   = remapped.count(tok.unk_id)
    token_count = len(remapped)

    enc = tok.encode(text)
    dec = tok.decode(enc["input_ids"])

    orig_words = set(text.lower().split())
    dec_words  = set(dec.lower().split())
    overlap    = len(orig_words & dec_words) / max(len(orig_words), 1)

    status = "✓" if overlap > 0.7 else "✗"
    print(f"  {text:<35} | {dec[:35]:<35} | {overlap:>6.0%} {status}")

    if overlap <= 0.7:
        all_passed = False
        unk_surfaces = [
            repr(tok._bloom_tok.decode([bid]))
            for bid, rid in zip(bloom_ids, remapped)
            if rid == tok.unk_id
        ]
        failures.append({
            "text":         text,
            "decoded":      dec,
            "overlap":      overlap,
            "unk_count":    unk_count,
            "unk_surfaces": unk_surfaces,
        })
        print(f"    ↳ UNK tokens ({unk_count}/{token_count}): {unk_surfaces}")

if failures:
    print(f"\n  Failure summary:")
    for f in failures:
        print(
            f"    '{f['text']}'\n"
            f"      overlap={f['overlap']:.0%}, "
            f"UNK surfaces: {f['unk_surfaces']}"
        )

# ── Assert — with full diagnostic message ─────────────────────
assert all_passed, (
    f"{len(failures)} round-trip test(s) failed.\n"
    f"\n"
    f"ROOT CAUSE:\n"
    f"  Critical tokens (=, %, $, digits) are mapping to UNK.\n"
    f"  This means the vocab file on disk is STALE — it was built\n"
    f"  before FORCED_VOCAB_TOKENS inflation was added.\n"
    f"\n"
    f"FIX (run in order):\n"
    f"  1. Run Cell 5-PRE  →  detects and deletes {VOCAB_FILE}\n"
    f"  2. Re-run Cell 3   →  tok.build_vocab(all_texts) rebuilds\n"
    f"  3. Re-run Cell 4   →  re-encode the example\n"
    f"  4. Re-run Cell 5   →  this test should now pass\n"
    f"\n"
    f"Failed texts: {[f['text'] for f in failures]}"
)

# ── Edge Case: All-PAD tensor ─────────────────────────────────
print("\n── Edge Case: Decode All-PAD Tensor ────────────────────")
all_pad = torch.zeros(MAX_SEQ_LEN, dtype=torch.long)
try:
    decoded_pad = tok.decode(all_pad)
    print(f"  All-PAD decoded: '{decoded_pad}'")
    assert decoded_pad == "", (
        f"All-PAD tensor should decode to empty string, got '{decoded_pad}'"
    )
    print(f"  ✓ Returns empty string (correct)")
except AssertionError:
    raise
except Exception as e:
    print(f"  ✗ Crashed on all-PAD input: {e}")
    raise

# ── Edge Case: BOS + EOS only ─────────────────────────────────
print("\n── Edge Case: Decode BOS+EOS Only ──────────────────────")
bos_eos_only    = torch.zeros(MAX_SEQ_LEN, dtype=torch.long)
bos_eos_only[0] = tok.bos_id
bos_eos_only[1] = tok.eos_id
try:
    decoded_bos_eos = tok.decode(bos_eos_only)
    print(f"  BOS+EOS decoded: '{decoded_bos_eos}'")
    assert decoded_bos_eos == "", (
        f"BOS+EOS-only tensor should decode to empty string, "
        f"got '{decoded_bos_eos}'"
    )
    print(f"  ✓ Returns empty string (correct)")
except AssertionError:
    raise
except Exception as e:
    print(f"  ✗ Crashed: {e}")
    raise

# ── Store results ─────────────────────────────────────────────
CELL5_RESULTS = {
    "decoded_example":        decoded,
    "overlap_ratio":          overlap_ratio,
    "all_round_trips_passed": all_passed,
}

print("\n" + "=" * 60)
print("✅ CELL 5 PASSED: Round-trip decode successful")
print(f"   Word overlap: {overlap_ratio:.1%} (threshold: 70%)")
print("=" * 60)

CELL 5: Decode Round-Trip Test

Original: 'If 3x + 9 = 21, find x'
Running tok.decode()...
Decoded:  'If 3x + 9 = 21, find x'
Type:     <class 'str'>

── Special Token Leakage Check ──────────────────────────
  ✓ No special token strings leaked into decoded text

── Content Preservation Check ───────────────────────────
  Key elements preserved: ['3', '9', '21', 'find', 'x']
  Key elements missing:   []

  Word-level overlap:
    Original words: ['+', '21', '3x', '9', '=', 'find', 'if', 'x']
    Decoded words:  ['+', '21', '3x', '9', '=', 'find', 'if', 'x']
    Overlap: 8/8 = 100.0%
  ✓ 100.0% word overlap (threshold: 70%)

── Multi-Text Round-Trip Test ───────────────────────────
  Original                            | Decoded                             | Overlap
  ----------------------------------- | ----------------------------------- | -------
  If 3x + 9 = 21, find x              | If 3x + 9 = 21, find x              |   100% ✓
  Step 1: Subtract 9 from both sides  | Step 1: Sub

In [6]:
# ============================================================
# CELL 6: Semantic Distinguishability Tests
#
# PURPOSE: Verify that different texts produce different token
#          IDs, and identical texts always produce the same IDs.
#
# WHY THIS IS CRITICAL:
#   The tokenizer output feeds directly into the context
#   encoder (Module 3). If two semantically different texts
#   produce the same token IDs, the encoder will produce
#   identical z_context vectors for them. The model cannot
#   learn to distinguish them, regardless of how large it is.
#
#   Conversely, if identical texts sometimes produce different
#   token IDs, training is unstable (same input, different
#   gradient signal).
#
# TESTS:
#   6.1: Different math problems → different IDs
#   6.2: Same text, called twice → identical IDs
#   6.3: Permuted word order → different IDs
#   6.4: Different numbers same structure → different IDs
#   6.5: Quantitative verification (Hamming distance analysis)
# ============================================================

print("=" * 60)
print("CELL 6: Semantic Distinguishability Tests")
print("=" * 60)

# ── Helper: Compute Hamming Distance Between ID Tensors ───────
def hamming_distance(ids_a: torch.Tensor, ids_b: torch.Tensor) -> int:
    """Number of positions where ids_a and ids_b differ."""
    return (ids_a != ids_b).sum().item()

def hamming_distance_ratio(ids_a: torch.Tensor, ids_b: torch.Tensor) -> float:
    """Fraction of positions where ids_a and ids_b differ."""
    return hamming_distance(ids_a, ids_b) / ids_a.numel()

# ── Test 6.1: Different Math Problems ─────────────────────────
print("\n── Test 6.1: Different Math Problems ────────────────────")

test_pairs_different = [
    ("3x + 9 = 21", "the cat sat on the mat"),
    ("Solve for x: 2x = 8", "Solve for y: 2y = 8"),
    ("If it rains, 15 people leave", "15% of $240"),
    ("Step 1: add 5", "Step 2: subtract 3"),
    ("The answer is 4", "The answer is 40"),
]

for text_a, text_b in test_pairs_different:
    ids_a = tok.encode(text_a)["input_ids"]
    ids_b = tok.encode(text_b)["input_ids"]
    dist = hamming_distance(ids_a, ids_b)
    ratio = hamming_distance_ratio(ids_a, ids_b)

    status = "✓" if dist > 0 else "✗"
    print(f"  {status} '{text_a[:30]}' vs '{text_b[:30]}'")
    print(f"     Hamming distance: {dist}/{config.MAX_SEQ_LEN} positions differ")

    assert dist > 0, (
        f"CRITICAL: Different texts produce identical token IDs!\n"
        f"  Text A: '{text_a}'\n"
        f"  Text B: '{text_b}'\n"
        f"  Both encode to: {ids_a[:10].tolist()}...\n"
        f"  The model cannot distinguish these inputs."
    )

# ── Test 6.2: Identical Texts → Identical IDs (Determinism) ───
print("\n── Test 6.2: Determinism (Identical Text) ───────────────")

identical_texts = [
    "If 3x + 9 = 21, find x",
    "The answer is 4",
    "Step 1: Subtract 9 from both sides",
    "",  # even empty string should be deterministic
    "x",
]

for text in identical_texts:
    enc_1 = tok.encode(text)
    enc_2 = tok.encode(text)
    enc_3 = tok.encode(text)  # triple check

    ids_equal_12 = torch.equal(enc_1["input_ids"], enc_2["input_ids"])
    ids_equal_23 = torch.equal(enc_2["input_ids"], enc_3["input_ids"])
    mask_equal_12 = torch.equal(enc_1["attention_mask"], enc_2["attention_mask"])

    all_equal = ids_equal_12 and ids_equal_23 and mask_equal_12
    status = "✓" if all_equal else "✗"
    print(f"  {status} '{text[:40]}' → same across 3 calls: {all_equal}")

    assert ids_equal_12, (
        f"Non-deterministic! Same text produced different IDs on calls 1 and 2.\n"
        f"  Text: '{text}'\n"
        f"  Call 1: {enc_1['input_ids'][:8].tolist()}\n"
        f"  Call 2: {enc_2['input_ids'][:8].tolist()}"
    )
    assert ids_equal_23, f"Non-deterministic on calls 2 and 3 for '{text}'"
    assert mask_equal_12, f"Attention mask is non-deterministic for '{text}'"

# ── Test 6.3: Word Order Sensitivity ─────────────────────────
print("\n── Test 6.3: Word Order Sensitivity ─────────────────────")

ordered_pairs = [
    ("John has 5 apples", "apples has 5 John"),
    ("Step 1: add 9", "Step 2: add 9"),
    ("3x + 9 = 21", "21 = 9 + 3x"),  # mathematically equivalent but different
]

for text_a, text_b in ordered_pairs:
    ids_a = tok.encode(text_a)["input_ids"]
    ids_b = tok.encode(text_b)["input_ids"]
    dist = hamming_distance(ids_a, ids_b)

    status = "✓" if dist > 0 else "⚠"
    print(f"  {status} '{text_a}' vs '{text_b}'")
    print(f"     Hamming distance: {dist} positions differ")

    # We expect different word order to produce different IDs
    # (may not always be the case if words hash to same positions)
    if dist == 0:
        print(f"     ⚠ Warning: Word order not reflected in token IDs")
        print(f"     (May be acceptable if encoder handles it)")

# ── Test 6.4: Number Sensitivity ─────────────────────────────
print("\n── Test 6.4: Number Sensitivity ─────────────────────────")

number_pairs = [
    ("The answer is 4", "The answer is 5"),
    ("3x + 9 = 21", "3x + 9 = 22"),
    ("15% discount", "16% discount"),
    ("$4.25", "$4.26"),
]

for text_a, text_b in number_pairs:
    ids_a = tok.encode(text_a)["input_ids"]
    ids_b = tok.encode(text_b)["input_ids"]
    dist = hamming_distance(ids_a, ids_b)

    status = "✓" if dist > 0 else "✗"
    print(f"  {status} '{text_a}' vs '{text_b}'")
    print(f"     Hamming distance: {dist} positions differ")

    assert dist > 0, (
        f"Different numbers produce same IDs!\n"
        f"  '{text_a}' and '{text_b}' are token-identical.\n"
        f"  The model cannot learn arithmetic distinctions."
    )

# ── Test 6.5: Quantitative Hamming Analysis ───────────────────
print("\n── Test 6.5: Quantitative Hamming Distance Analysis ─────")

# Build a set of related and unrelated texts
related_texts = [
    "3x + 9 = 21",
    "3x + 9 = 22",
    "3x + 10 = 21",
    "4x + 9 = 21",
]

unrelated_texts = [
    "The bakery sells bread",
    "A dog runs in the park",
    "She bought groceries yesterday",
    "It rained all morning",
]

print("\n  Related math texts (expect moderate similarity):")
for i in range(len(related_texts)):
    for j in range(i + 1, len(related_texts)):
        ids_i = tok.encode(related_texts[i])["input_ids"]
        ids_j = tok.encode(related_texts[j])["input_ids"]
        dist = hamming_distance(ids_i, ids_j)
        print(f"    '{related_texts[i]}' vs '{related_texts[j]}'")
        print(f"      Positions different: {dist} ({dist/config.MAX_SEQ_LEN:.1%})")

print("\n  Math vs. unrelated texts (expect high distance):")
math_example = tok.encode("3x + 9 = 21")["input_ids"]
for unrelated in unrelated_texts:
    ids_unrelated = tok.encode(unrelated)["input_ids"]
    dist = hamming_distance(math_example, ids_unrelated)
    print(f"    '3x + 9 = 21' vs '{unrelated}'")
    print(f"      Positions different: {dist} ({dist/config.MAX_SEQ_LEN:.1%})")

# Store results
CELL6_RESULTS = {
    "all_different_pairs_passed": True,
    "determinism_verified": True,
}

print("\n" + "=" * 60)
print("✅ CELL 6 PASSED: Semantic distinguishability verified")
print("   Different texts → different IDs")
print("   Identical texts → identical IDs (deterministic)")
print("=" * 60)

2026-06-12 15:38:08,407 | modules.tokenizer | WARNING | Empty string passed to encode(). Returning minimal BOS+EOS tensor.
2026-06-12 15:38:08,410 | modules.tokenizer | WARNING | Empty string passed to encode(). Returning minimal BOS+EOS tensor.
2026-06-12 15:38:08,411 | modules.tokenizer | WARNING | Empty string passed to encode(). Returning minimal BOS+EOS tensor.


CELL 6: Semantic Distinguishability Tests

── Test 6.1: Different Math Problems ────────────────────
  ✓ '3x + 9 = 21' vs 'the cat sat on the mat'
     Hamming distance: 6/256 positions differ
  ✓ 'Solve for x: 2x = 8' vs 'Solve for y: 2y = 8'
     Hamming distance: 6/256 positions differ
  ✓ 'If it rains, 15 people leave' vs '15% of $240'
     Hamming distance: 8/256 positions differ
  ✓ 'Step 1: add 5' vs 'Step 2: subtract 3'
     Hamming distance: 3/256 positions differ
  ✓ 'The answer is 4' vs 'The answer is 40'
     Hamming distance: 1/256 positions differ

── Test 6.2: Determinism (Identical Text) ───────────────
  ✓ 'If 3x + 9 = 21, find x' → same across 3 calls: True
  ✓ 'The answer is 4' → same across 3 calls: True
  ✓ 'Step 1: Subtract 9 from both sides' → same across 3 calls: True
  ✓ '' → same across 3 calls: True
  ✓ 'x' → same across 3 calls: True

── Test 6.3: Word Order Sensitivity ─────────────────────
  ✓ 'John has 5 apples' vs 'apples has 5 John'
     Hamming distanc

In [7]:
# ============================================================
# CELL 7: Batch Encoding Test
#
# PURPOSE: Verify batch_encode() produces correctly shaped
#          output and that each row is identical to what
#          individual encode() calls would produce.
#
# WHY CONSISTENCY MATTERS:
#   Training uses batch_encode(). Evaluation might use
#   individual encode(). If they produce different results,
#   train/eval metrics are incomparable.
#
# TESTS:
#   7.1: Output shape is exactly (B, MAX_SEQ_LEN)
#   7.2: Each row matches individual encode() output
#   7.3: Variable-length texts all fit in the batch
#   7.4: Batch with size=1 works (edge case for eval)
#   7.5: Large batch doesn't OOM (batch_size=64)
#   7.6: Empty batch raises gracefully or returns empty tensor
# ============================================================

print("=" * 60)
print("CELL 7: Batch Encoding Test")
print("=" * 60)

# ── Standard Test Batch ───────────────────────────────────────
# Deliberately chosen to have very different lengths
texts_batch = [
    "If 3x + 9 = 21, find x",                                    # ~7 content tokens
    "A train travels at 60 mph for 2.5 hours. What is the distance?",  # ~12 content tokens
    "x",                                                          # 1 content token (minimum)
    "The store sold 42 items at $3.50 each and gave a 15% "
    "discount to all customers who bought more than 10 items.",   # ~25 content tokens
    "",                                                           # 0 content tokens (edge)
    "Step 1: Subtract 9 from both sides. 3x = 12. "
    "Step 2: Divide both sides by 3. x = 4. "
    "Therefore x = 4.",                                           # ~20 content tokens
]
B = len(texts_batch)

# ── Test 7.1: Output Shape ────────────────────────────────────
print(f"\n── Test 7.1: Output Shape ───────────────────────────────")
print(f"  Batch size: {B} texts")

batch_result = tok.batch_encode(texts_batch)

expected_ids_shape = (B, config.MAX_SEQ_LEN)
expected_mask_shape = (B, config.MAX_SEQ_LEN)

ids_shape = batch_result["input_ids"].shape
mask_shape = batch_result["attention_mask"].shape

print(f"  input_ids shape:      {ids_shape}  (expected {expected_ids_shape})")
print(f"  attention_mask shape: {mask_shape}  (expected {expected_mask_shape})")

assert ids_shape == expected_ids_shape, (
    f"Wrong batch input_ids shape!\n"
    f"  Expected: {expected_ids_shape}\n"
    f"  Got:      {ids_shape}"
)
assert mask_shape == expected_mask_shape, (
    f"Wrong batch attention_mask shape!\n"
    f"  Expected: {expected_mask_shape}\n"
    f"  Got:      {mask_shape}"
)
print(f"  ✓ Both tensors have correct shape")

# Verify dtypes
assert batch_result["input_ids"].dtype == torch.long, (
    f"Batch input_ids must be LongTensor, got {batch_result['input_ids'].dtype}"
)
assert batch_result["attention_mask"].dtype == torch.long, (
    f"Batch attention_mask must be LongTensor, got {batch_result['attention_mask'].dtype}"
)
print(f"  ✓ Both tensors have dtype=torch.long")

# ── Test 7.2: Consistency With Individual encode() ────────────
print(f"\n── Test 7.2: Consistency With Individual encode() ───────")
print(f"  Comparing each batch row to individual encode() result:")

all_consistent = True
for i, text in enumerate(texts_batch):
    # Individual encode
    individual = tok.encode(text)
    ind_ids = individual["input_ids"]
    ind_mask = individual["attention_mask"]

    # Batch row
    batch_ids = batch_result["input_ids"][i]
    batch_mask = batch_result["attention_mask"][i]

    ids_match = torch.equal(ind_ids, batch_ids)
    mask_match = torch.equal(ind_mask, batch_mask)

    status = "✓" if (ids_match and mask_match) else "✗"
    print(f"  {status} Row {i}: '{text[:40]}'")

    if not ids_match:
        all_consistent = False
        print(f"      MISMATCH in input_ids!")
        print(f"      Individual: {ind_ids[:6].tolist()}")
        print(f"      Batch row:  {batch_ids[:6].tolist()}")

    if not mask_match:
        all_consistent = False
        print(f"      MISMATCH in attention_mask!")

assert all_consistent, (
    "batch_encode() results inconsistent with encode().\n"
    "Training data will differ from what you'd get encoding individually."
)
print(f"  ✓ All rows match individual encode() output")

# ── Test 7.3: Variable Lengths ────────────────────────────────
print(f"\n── Test 7.3: Variable Length Handling ───────────────────")

real_counts = batch_result["attention_mask"].sum(dim=1).tolist()

print(f"  {'Text':<55} | {'Real tokens':>11}")
print(f"  {'-'*55} | {'-'*11}")
for i, (text, count) in enumerate(zip(texts_batch, real_counts)):
    display = text[:52] + "..." if len(text) > 55 else text
    print(f"  {display:<55} | {count:>11}")

# Each row must have at least BOS + EOS = 2 real tokens
for i, count in enumerate(real_counts):
    assert count >= 2, (
        f"Row {i} has only {count} real tokens.\n"
        f"  Minimum is 2 (BOS + EOS).\n"
        f"  Text: '{texts_batch[i]}'"
    )
    assert count <= config.MAX_SEQ_LEN, (
        f"Row {i} has {count} real tokens, exceeding MAX_SEQ_LEN={config.MAX_SEQ_LEN}"
    )

print(f"\n  Token counts range: [{min(real_counts)}, {max(real_counts)}]")
print(f"  ✓ All rows have valid token counts")

# ── Test 7.4: Batch Size = 1 ─────────────────────────────────
print(f"\n── Test 7.4: Batch Size = 1 ─────────────────────────────")
single_batch = tok.batch_encode(["If 3x + 9 = 21, find x"])
assert single_batch["input_ids"].shape == (1, config.MAX_SEQ_LEN), (
    f"Batch size 1 failed: shape is {single_batch['input_ids'].shape}"
)
assert single_batch["attention_mask"].shape == (1, config.MAX_SEQ_LEN)
print(f"  ✓ batch_size=1 shape: {single_batch['input_ids'].shape}")

# Verify row matches individual encode
ind = tok.encode("If 3x + 9 = 21, find x")
assert torch.equal(single_batch["input_ids"][0], ind["input_ids"]), (
    "batch_encode([text]) row 0 ≠ encode(text) for same text"
)
print(f"  ✓ batch_size=1 row matches individual encode()")

# ── Test 7.5: Large Batch (No OOM) ───────────────────────────
print(f"\n── Test 7.5: Large Batch (B=64) ────────────────────────")
large_batch_texts = [
    f"Solve for x: {i}x + {i*2} = {i*5}"
    for i in range(1, 65)
]

try:
    large_result = tok.batch_encode(large_batch_texts)
    assert large_result["input_ids"].shape == (64, config.MAX_SEQ_LEN)
    assert large_result["attention_mask"].shape == (64, config.MAX_SEQ_LEN)
    print(f"  ✓ Batch size 64: shape {large_result['input_ids'].shape}")
    mem_bytes = large_result["input_ids"].element_size() * large_result["input_ids"].numel()
    print(f"  Memory for input_ids: {mem_bytes / 1024:.1f} KB")
except torch.cuda.OutOfMemoryError:
    print(f"  ⚠ OOM at batch_size=64 — tokenizer uses GPU?")
    raise

# ── Test 7.6: Empty Batch ─────────────────────────────────────
print(f"\n── Test 7.6: Empty Batch [] ─────────────────────────────")
try:
    empty_result = tok.batch_encode([])
    # Should return tensors with 0 rows or raise gracefully
    if "input_ids" in empty_result:
        print(f"  Returns empty tensor: {empty_result['input_ids'].shape}")
        assert empty_result["input_ids"].shape[0] == 0, (
            "Empty batch should return 0-row tensor"
        )
    print(f"  ✓ Empty batch handled gracefully")
except (ValueError, RuntimeError) as e:
    print(f"  ✓ Empty batch raises {type(e).__name__}: {e} (acceptable)")
except Exception as e:
    print(f"  ✗ Unexpected error on empty batch: {type(e).__name__}: {e}")
    raise

# Store results
CELL7_RESULTS = {
    "batch_result": batch_result,
    "real_counts": real_counts,
    "consistency_verified": all_consistent,
}

print("\n" + "=" * 60)
print("✅ CELL 7 PASSED: Batch encoding correct")
print(f"   Shape: ({B}, {config.MAX_SEQ_LEN}), consistent with individual encode()")
print("=" * 60)

2026-06-12 15:38:13,612 | modules.tokenizer | WARNING | Empty string passed to encode(). Returning minimal BOS+EOS tensor.
2026-06-12 15:38:13,621 | modules.tokenizer | WARNING | Empty string passed to encode(). Returning minimal BOS+EOS tensor.


CELL 7: Batch Encoding Test

── Test 7.1: Output Shape ───────────────────────────────
  Batch size: 6 texts
  input_ids shape:      torch.Size([6, 256])  (expected (6, 256))
  attention_mask shape: torch.Size([6, 256])  (expected (6, 256))
  ✓ Both tensors have correct shape
  ✓ Both tensors have dtype=torch.long

── Test 7.2: Consistency With Individual encode() ───────
  Comparing each batch row to individual encode() result:
  ✓ Row 0: 'If 3x + 9 = 21, find x'
  ✓ Row 1: 'A train travels at 60 mph for 2.5 hours.'
  ✓ Row 2: 'x'
  ✓ Row 3: 'The store sold 42 items at $3.50 each an'
  ✓ Row 4: ''
  ✓ Row 5: 'Step 1: Subtract 9 from both sides. 3x ='
  ✓ All rows match individual encode() output

── Test 7.3: Variable Length Handling ───────────────────
  Text                                                    | Real tokens
  ------------------------------------------------------- | -----------
  If 3x + 9 = 21, find x                                  |          11
  A train travels a

In [9]:
# ── Test 8.3: String Longer Than MAX_SEQ_LEN ─────────────────
print("\n── 8.3: String Longer Than MAX_SEQ_LEN ─────────────────")

# WHY THE OLD APPROACH FAILED:
#   "solve for x and find the value " * 30 produces only 214
#   real tokens, not 256+. Two reasons:
#
#   1. BLOOM's BPE merges repeated substrings aggressively.
#      "solve for x" may become fewer tokens than you expect.
#
#   2. Some words map to unk_id but unk_id still counts as a
#      real token — so word count != token count.
#
# CORRECT APPROACH:
#   Generate the long text, encode it WITHOUT the 256-cap,
#   verify that raw token count > MAX_SEQ_LEN, THEN test
#   that _pad_and_mask truncates it correctly.

def _count_raw_tokens(text: str) -> int:
    """
    Count how many reduced IDs (including BOS + EOS) encode()
    would produce BEFORE truncation.
    Useful for guaranteeing a text actually overflows MAX_SEQ_LEN.
    """
    bloom_ids   = tok._bloom_tok.encode(text, add_special_tokens=False)
    reduced_ids = tok._remap_ids(bloom_ids)
    # +2 for BOS and EOS that encode() prepends/appends
    return len(reduced_ids) + 2


# ── Build a long text that PROVABLY overflows ─────────────────
# Strategy: use short common words so they don't become UNK,
# and repeat until raw token count exceeds MAX_SEQ_LEN.

base_phrase = "the cat sat on the mat and the dog ran to the car "
long_text   = base_phrase

while _count_raw_tokens(long_text) <= config.MAX_SEQ_LEN:
    long_text += base_phrase

raw_token_count = _count_raw_tokens(long_text)

print(f"  Long text stats:")
print(f"    Characters:         {len(long_text)}")
print(f"    Words:              {len(long_text.split())}")
print(f"    Raw tokens (BOS+content+EOS): {raw_token_count}")
print(f"    MAX_SEQ_LEN:        {config.MAX_SEQ_LEN}")
print(f"    Overflows by:       {raw_token_count - config.MAX_SEQ_LEN} tokens")

assert raw_token_count > config.MAX_SEQ_LEN, (
    f"long_text produces only {raw_token_count} tokens — "
    f"does not exceed MAX_SEQ_LEN={config.MAX_SEQ_LEN}.\n"
    f"The while loop above should have prevented this."
)
print(f"  ✓ Confirmed: raw token count ({raw_token_count}) > MAX_SEQ_LEN ({config.MAX_SEQ_LEN})")

# ── Now encode and verify truncation ─────────────────────────
r_long = test_edge_case(
    "long string",
    text=long_text,
    expect_min_real_tokens=config.MAX_SEQ_LEN,  # fully packed
    expect_max_real_tokens=config.MAX_SEQ_LEN,  # no overflow
)
n_real_long = r_long["n_real"]

print(f"\n  After encode():")
print(f"    Real tokens:  {n_real_long} (expected {config.MAX_SEQ_LEN})")
print(f"    Padding:      {config.MAX_SEQ_LEN - n_real_long} positions")

assert n_real_long == config.MAX_SEQ_LEN, (
    f"Expected exactly MAX_SEQ_LEN={config.MAX_SEQ_LEN} real tokens "
    f"after truncation, got {n_real_long}.\n"
    f"Check _pad_and_mask() truncation logic."
)

# ── Verify EOS is preserved at the truncation boundary ────────
last_id = r_long["ids"][config.MAX_SEQ_LEN - 1].item()
print(f"\n  Truncation boundary check:")
print(f"    ids[{config.MAX_SEQ_LEN - 1}] = {last_id}  (expected EOS={tok.eos_id})")

assert last_id == tok.eos_id, (
    f"Truncated sequence must end with EOS (id={tok.eos_id}).\n"
    f"  Got: {last_id}\n"
    f"  _pad_and_mask() should replace the last token with EOS "
    f"when truncating."
)
print(f"  ✓ EOS preserved at position {config.MAX_SEQ_LEN - 1}")

# ── Verify no padding exists (sequence is fully packed) ───────
pad_count = (r_long["ids"] == tok.pad_id).sum().item()
print(f"\n  Padding check:")
print(f"    PAD tokens in output: {pad_count} (expected 0)")

assert pad_count == 0, (
    f"Fully-packed sequence should have 0 PAD tokens, "
    f"found {pad_count}.\n"
    f"  If real tokens < MAX_SEQ_LEN, the overflow text wasn't "
    f"long enough. The while loop above should handle this."
)
print(f"  ✓ Zero padding — sequence is fully packed")

# ── Verify attention mask is all-ones ─────────────────────────
mask_sum = r_long["mask"].sum().item()
print(f"\n  Attention mask check:")
print(f"    mask.sum() = {mask_sum} (expected {config.MAX_SEQ_LEN})")

assert mask_sum == config.MAX_SEQ_LEN, (
    f"Fully-packed sequence should have all-ones mask.\n"
    f"  mask.sum() = {mask_sum}, expected {config.MAX_SEQ_LEN}"
)
print(f"  ✓ Attention mask is all-ones")

print(f"\n  ✓ Long string correctly truncated to MAX_SEQ_LEN={config.MAX_SEQ_LEN}")


── 8.3: String Longer Than MAX_SEQ_LEN ─────────────────
  Long text stats:
    Characters:         1000
    Words:              260
    Raw tokens (BOS+content+EOS): 263
    MAX_SEQ_LEN:        256
    Overflows by:       7 tokens
  ✓ Confirmed: raw token count (263) > MAX_SEQ_LEN (256)

  After encode():
    Real tokens:  256 (expected 256)
    Padding:      0 positions

  Truncation boundary check:
    ids[255] = 2  (expected EOS=2)
  ✓ EOS preserved at position 255

  Padding check:
    PAD tokens in output: 0 (expected 0)
  ✓ Zero padding — sequence is fully packed

  Attention mask check:
    mask.sum() = 256 (expected 256)
  ✓ Attention mask is all-ones

  ✓ Long string correctly truncated to MAX_SEQ_LEN=256


In [ ]:
# ============================================================
# CELL 8: Edge Cases
#
# PURPOSE: Test the inputs that break production systems.
#          These are not exotic — they appear in real datasets.
#
# WHAT WE VERIFY:
#   Every test checks the same six structural invariants:
#     I1: output shape is exactly (MAX_SEQ_LEN,)
#     I2: dtype is torch.long for both tensors
#     I3: position 0 is always BOS
#     I4: last real token is always EOS
#     I5: all padding positions are exactly 0
#     I6: all token IDs are in [0, VOCAB_SIZE)
#
# EDGE CASES COVERED:
#   8.1  Empty string          ""
#   8.2  Single character      "x"
#   8.3  Overflow (> MAX_SEQ_LEN tokens) — must truncate cleanly
#   8.4  Unicode math symbols  √ × ÷ π ≤ ≥ ≠ ²
#   8.5  Currency / percent    $ £ € %
#   8.6  Whitespace variants   \n \t \r and mixed spaces
#   8.7  Numeric-only strings  integers, decimals, expressions
#   8.8  Punctuation-heavy     ... ??? () []
#   8.9  Mixed batch           all of the above in one batch_encode()
#
# WHY EDGE CASES MATTER:
#   GSM8K contains all of the above.  A tokenizer that crashes on
#   "√16 = 4" or silently corrupts "$240 at 15%" will produce
#   broken training examples that poison model learning.
#
# TEST 8.3 DESIGN NOTE:
#   We do NOT guess how many tokens a repeated phrase produces.
#   Instead we call _count_raw_tokens() in a while-loop until
#   we KNOW the text overflows MAX_SEQ_LEN before truncation.
#   This makes the test corpus-independent.
# ============================================================

from typing import Dict, Optional

print("=" * 60)
print("CELL 8: Edge Cases")
print("=" * 60)


# ── Helper: single-case encoder + verifier ────────────────────
# Used by every test below.  Returns a result dict so callers
# can inspect n_real, ids, mask without re-encoding.
def test_edge_case(
    name: str,
    text: str,
    expect_min_real_tokens: int = 2,
    expect_max_real_tokens: Optional[int] = None,
    expect_eos_at_end: bool = True,
    expect_no_crash: bool = True,
) -> Dict:
    """
    Encode `text` and assert all six structural invariants hold.

    Args:
        name:                   Human-readable label shown in output.
        text:                   Raw string to encode.
        expect_min_real_tokens: Minimum acceptable real-token count
                                (mask==1 positions).  Default 2 = BOS+EOS.
        expect_max_real_tokens: Upper bound on real-token count.
                                None means unconstrained.
        expect_eos_at_end:      If True, assert ids[n_real-1] == eos_id.
        expect_no_crash:        If True, re-raise any exception as
                                AssertionError so the cell fails loudly.

    Returns:
        Dict with keys: name, text, passed, error, n_real, ids, mask.
    """
    result_info = {"name": name, "text": text, "passed": False, "error": None}

    try:
        enc  = tok.encode(text)
        ids  = enc["input_ids"]
        mask = enc["attention_mask"]

        # I1: shape
        assert ids.shape  == (config.MAX_SEQ_LEN,), f"Wrong shape: {ids.shape}"
        assert mask.shape == (config.MAX_SEQ_LEN,), f"Wrong mask shape: {mask.shape}"

        # Count real vs padding positions
        n_real = mask.sum().item()

        # Token-count bounds
        assert n_real >= expect_min_real_tokens, (
            f"Too few real tokens: {n_real} < {expect_min_real_tokens}"
        )
        if expect_max_real_tokens is not None:
            assert n_real <= expect_max_real_tokens, (
                f"Too many real tokens: {n_real} > {expect_max_real_tokens}"
            )

        # I3: BOS at position 0
        assert ids[0].item() == tok.bos_id, (
            f"Position 0 is {ids[0].item()}, expected BOS={tok.bos_id}"
        )

        # I4: EOS at last real position
        if expect_eos_at_end and n_real > 0:
            last_real = ids[n_real - 1].item()
            assert last_real == tok.eos_id, (
                f"Last real token is {last_real}, expected EOS={tok.eos_id}"
            )

        # I5: padding is exactly zero
        if n_real < config.MAX_SEQ_LEN:
            padding_sum = ids[n_real:].sum().item()
            assert padding_sum == 0, f"Non-zero padding: {padding_sum}"

        # I6: ID range
        assert ids.max().item() < config.VOCAB_SIZE, (
            f"ID {ids.max().item()} >= VOCAB_SIZE {config.VOCAB_SIZE}"
        )
        assert ids.min().item() >= 0, f"Negative ID: {ids.min().item()}"

        result_info.update({
            "passed": True,
            "n_real": n_real,
            "shape":  ids.shape,
            "bos_ok": True,
            "eos_ok": True,
            "ids":    ids,
            "mask":   mask,
        })

    except Exception as e:
        result_info["error"] = str(e)
        if expect_no_crash:
            raise AssertionError(
                f"Edge case '{name}' failed with: {e}\n"
                f"  Text: '{text[:80]}'"
            ) from e

    return result_info


# ── Helper: raw token count before truncation ─────────────────
# Required by Test 8.3 to guarantee the input actually overflows.
# Counts BLOOM tokens remapped to reduced IDs, plus BOS and EOS.
# Does NOT call encode() so no truncation occurs.
def _count_raw_tokens(text: str) -> int:
    """
    Return the number of tokens encode() would produce BEFORE
    any truncation or padding is applied.

    Formula: len(remap(bloom_tokenize(text))) + 2
             (+2 accounts for BOS and EOS that encode() prepends/appends)

    Used in Test 8.3 to build a text that provably overflows
    MAX_SEQ_LEN, making the truncation assertion corpus-independent.
    """
    bloom_ids   = tok._bloom_tok.encode(text, add_special_tokens=False)
    reduced_ids = tok._remap_ids(bloom_ids)
    return len(reduced_ids) + 2   # +2 for BOS and EOS


# ── Test 8.1: Empty String ────────────────────────────────────
# WHY: Empty strings appear when GSM8K examples have blank
#      questions or answers after stripping whitespace.
#      encode("") must not crash and must return a valid tensor.
# EXPECTED: BOS + EOS = 2 real tokens (no content between them).
print("\n── 8.1: Empty String ────────────────────────────────────")
r = test_edge_case(
    "empty string",
    text="",
    expect_min_real_tokens=2,   # at minimum BOS + EOS
    expect_max_real_tokens=3,   # some tokenizers insert an empty token
)
print(f"  Real tokens: {r['n_real']}  (BOS+EOS=2, or BOS+empty+EOS=3)")
print(f"  IDs: {r['ids'][:5].tolist()}...")
print(f"  ✓ Empty string handled correctly")


# ── Test 8.2: Single Character ────────────────────────────────
# WHY: "x", "n", "a" appear as standalone variable names in
#      algebra problems.  Must produce BOS + token + EOS.
# EXPECTED: 3 real tokens (4 if BLOOM splits the char into bytes).
print("\n── 8.2: Single Character 'x' ────────────────────────────")
r = test_edge_case(
    "single char",
    text="x",
    expect_min_real_tokens=3,   # BOS + x + EOS
    expect_max_real_tokens=4,   # BOS + byte1 + byte2 + EOS (fallback)
)
print(f"  Real tokens: {r['n_real']}")
print(f"  IDs: {r['ids'][:6].tolist()}")
print(f"  ✓ Single character handled")


# ── Test 8.3: String Longer Than MAX_SEQ_LEN ─────────────────
# WHY: Long multi-step GSM8K solutions can exceed 256 tokens.
#      _pad_and_mask() must truncate to exactly MAX_SEQ_LEN and
#      replace the last token with EOS so the model always sees
#      a sequence boundary even in truncated inputs.
#
# APPROACH: We do NOT hard-code a word count.  Instead we call
#   _count_raw_tokens() in a while-loop, appending base_phrase
#   until the raw count provably exceeds MAX_SEQ_LEN.
#   This makes the test independent of tokenizer version and
#   corpus composition.
#
# BASE PHRASE CHOICE: short, common English words that are
#   virtually guaranteed to be in the vocab (not UNK), so the
#   raw token count reflects actual sequence length faithfully.
print("\n── 8.3: String Longer Than MAX_SEQ_LEN ─────────────────")

base_phrase = "the cat sat on the mat and the dog ran to the car "
long_text   = base_phrase

# Keep appending until raw token count (pre-truncation) exceeds cap
while _count_raw_tokens(long_text) <= config.MAX_SEQ_LEN:
    long_text += base_phrase

raw_count = _count_raw_tokens(long_text)
print(f"  Built overflow text:")
print(f"    Words:          {len(long_text.split())}")
print(f"    Raw tokens:     {raw_count}  (before truncation)")
print(f"    MAX_SEQ_LEN:    {config.MAX_SEQ_LEN}")
print(f"    Overflow by:    {raw_count - config.MAX_SEQ_LEN} tokens")

# After encode(), output must be exactly MAX_SEQ_LEN real tokens
r_long = test_edge_case(
    "long string",
    text=long_text,
    expect_min_real_tokens=config.MAX_SEQ_LEN,
    expect_max_real_tokens=config.MAX_SEQ_LEN,
)
print(f"  After encode(): {r_long['n_real']} real tokens ✓")

# Additional checks specific to truncation
last_id   = r_long["ids"][config.MAX_SEQ_LEN - 1].item()
pad_count = (r_long["ids"] == tok.pad_id).sum().item()
mask_sum  = r_long["mask"].sum().item()

assert last_id == tok.eos_id, (
    f"Truncated sequence must end with EOS={tok.eos_id}, got {last_id}.\n"
    f"  _pad_and_mask() must overwrite ids[-1] with EOS when truncating."
)
assert pad_count == 0, (
    f"Fully-packed sequence should have 0 PAD tokens, found {pad_count}."
)
assert mask_sum == config.MAX_SEQ_LEN, (
    f"mask.sum() should equal MAX_SEQ_LEN={config.MAX_SEQ_LEN}, "
    f"got {mask_sum}."
)

print(f"  EOS at position {config.MAX_SEQ_LEN - 1}: {last_id} ✓")
print(f"  PAD tokens: {pad_count} ✓")
print(f"  mask.sum(): {mask_sum} ✓")
print(f"  ✓ Long string truncated correctly")


# ── Test 8.4: Mathematical Symbols ───────────────────────────
# WHY: GSM8K uses Unicode math notation.  BLOOM's BPE may split
#      "√" into multiple byte-fallback tokens or map them to UNK.
#      We verify: no crash, valid shape, BOS/EOS present.
#      We do NOT assert round-trip fidelity here — Unicode symbols
#      outside Latin-1 legitimately become UNK and that is
#      acceptable as long as the tensor structure is valid.
print("\n── 8.4: Mathematical Symbols ────────────────────────────")
math_cases = [
    ("sqrt notation",       "√16 = 4"),
    ("multiplication sign", "4 × 3 = 12"),
    ("division sign",       "12 ÷ 4 = 3"),
    ("pi",                  "π ≈ 3.14159"),
    ("not equal",           "x ≠ 0"),
    ("less/greater or eq",  "x ≤ 5 and y ≥ 2"),
    ("superscript 2",       "x² + y² = r²"),
    ("combined",            "√(x²+y²) ≤ r, where r = 5"),
]
for name, text in math_cases:
    r = test_edge_case(name, text, expect_min_real_tokens=3)
    print(f"  ✓ {name}: '{text}' → {r['n_real']} tokens")


# ── Test 8.5: Currency and Percentages ───────────────────────
# WHY: GSM8K word problems frequently involve dollar amounts,
#      percentages, and mixed currency expressions.
#      These were the tokens that originally failed (Cell 5).
#      FORCED_VOCAB_TOKENS guarantees "$", "%", and common
#      merged forms ("15%", "124") are in the vocab.
#      We verify they encode without crashing and produce
#      valid tensor structure.
print("\n── 8.5: Currency and Percentages ────────────────────────")
currency_cases = [
    ("USD",      "$4.25"),
    ("GBP",      "£12.50"),
    ("EUR",      "€99.99"),
    ("percent",  "15%"),
    ("combined", "$240 at 15% discount = $204.00"),
    ("commas",   "$12,500.00"),
    ("negative", "-$50.00"),
]
for name, text in currency_cases:
    r = test_edge_case(name, text, expect_min_real_tokens=3)
    print(f"  ✓ {name}: '{text}' → {r['n_real']} tokens")


# ── Test 8.6: Newlines and Tabs ───────────────────────────────
# WHY: GSM8K answer strings use "\n" as a step separator.
#      Example: "Step 1: ...\nStep 2: ...\n#### 4"
#      BLOOM's tokenizer handles \n as a regular character and
#      may merge it with adjacent tokens.  We verify no crash.
print("\n── 8.6: Newlines and Tabs ───────────────────────────────")
whitespace_cases = [
    ("newline",         "Step 1: add 9\nStep 2: divide by 3"),
    ("tab",             "Step 1:\tSubtract 9"),
    ("multi-newline",   "Q: Find x\n\nA: x = 4"),
    ("carriage return", "solve\rfor x"),
    ("mixed ws",        "  spaces  before  "),
]
for name, text in whitespace_cases:
    r = test_edge_case(name, text, expect_min_real_tokens=3)
    print(f"  ✓ {name}: repr='{repr(text[:40])}' → {r['n_real']} tokens")


# ── Test 8.7: Numeric-Only Strings ───────────────────────────
# WHY: Answers to GSM8K problems are often pure numbers.
#      "42", "3.14", "-7" must each produce at least BOS+token+EOS.
#      Digits and common two/three-digit numbers are in
#      FORCED_VOCAB_TOKENS so they are guaranteed to be in vocab.
print("\n── 8.7: Numeric-Only Strings ────────────────────────────")
number_cases = [
    ("integer",      "42"),
    ("decimal",      "3.14159"),
    ("long decimal", "0.001234567890"),
    ("large number", "1234567890"),
    ("negative",     "-42"),
    ("equation",     "3 + 9 = 21"),
    ("expression",   "3.5 * 4 + 2.5"),
]
for name, text in number_cases:
    r = test_edge_case(name, text, expect_min_real_tokens=3)
    print(f"  ✓ {name}: '{text}' → {r['n_real']} tokens")


# ── Test 8.8: Punctuation-Heavy Text ─────────────────────────
# WHY: Strings like "..." or "???" appear in poorly formatted
#      dataset entries.  They must not crash encode() or produce
#      tensors with invalid IDs.
#      expect_min_real_tokens=2 because a string of only punctuation
#      may reduce to BOS+EOS if all punctuation maps to UNK.
print("\n── 8.8: Punctuation-Heavy Text ─────────────────────────")
punct_cases = [
    ("periods",        "..."),
    ("question marks", "???"),
    ("mixed punct",    "If x = 4, then: 3x + 9 = 21."),
    ("parentheses",    "(3 + 9) × 2 = 24"),
    ("brackets",       "[Step 1] Subtract 9"),
]
for name, text in punct_cases:
    r = test_edge_case(name, text, expect_min_real_tokens=2)
    print(f"  ✓ {name}: '{text}' → {r['n_real']} tokens")


# ── Test 8.9: Batch With Mixed Edge Cases ─────────────────────
# WHY: batch_encode() must handle a list where some entries are
#      empty, some are single chars, and some are normal text —
#      all in one call.  Every row in the output batch must have
#      the correct shape and at least BOS+EOS.
#
# WHAT WE CHECK:
#   - Output shape is (B, MAX_SEQ_LEN) — no ragged tensors
#   - Every row has attention_mask.sum() >= 2 (BOS + EOS minimum)
print("\n── 8.9: Batch Encoding With Edge Cases ──────────────────")
edge_batch = [
    "",                    # 8.1 repr — empty
    "x",                   # 8.2 repr — single char
    "$4.25",               # 8.5 repr — currency
    "√16 = 4",             # 8.4 repr — math symbol
    "step 1\nstep 2",      # 8.6 repr — newline
    "normal math problem", # baseline — no edge case
]

edge_batch_result = tok.batch_encode(edge_batch)
expected_shape    = (len(edge_batch), config.MAX_SEQ_LEN)

assert edge_batch_result["input_ids"].shape == expected_shape, (
    f"Edge batch shape wrong: {edge_batch_result['input_ids'].shape} "
    f"(expected {expected_shape}).\n"
    f"  batch_encode() must return a fixed rectangular tensor — "
    f"no ragged rows."
)

counts = edge_batch_result["attention_mask"].sum(dim=1).tolist()
print(f"  Batch shape: {edge_batch_result['input_ids'].shape} ✓")
print(f"  Real tokens per row: {[int(c) for c in counts]}")

assert all(c >= 2 for c in counts), (
    f"Every row must have at least BOS+EOS (count >= 2).\n"
    f"  Row counts: {counts}\n"
    f"  A row with count < 2 means BOS or EOS was dropped."
)
print(f"  ✓ All rows have valid token counts")


# ── Summary ───────────────────────────────────────────────────
print("\n── Edge Case Summary ────────────────────────────────────")
print("  ✓ 8.1 Empty string            → BOS+EOS returned, no crash")
print("  ✓ 8.2 Single character        → BOS + token + EOS")
print("  ✓ 8.3 Longer than MAX_SEQ_LEN → truncated, EOS preserved")
print("  ✓ 8.4 Mathematical symbols    → √ × ÷ π ≤ ² no crash")
print("  ✓ 8.5 Currency and percentages→ $ £ € % valid tensors")
print("  ✓ 8.6 Whitespace variants     → \\n \\t \\r handled")
print("  ✓ 8.7 Numeric strings         → integers, decimals, expressions")
print("  ✓ 8.8 Punctuation-heavy       → ... ??? () [] no crash")
print("  ✓ 8.9 Mixed batch             → rectangular shape, all rows valid")

print("\n" + "=" * 60)
print("✅ CELL 8 PASSED: All edge cases handled correctly")
print("=" * 60)

2026-06-12 15:40:20,579 | modules.tokenizer | WARNING | Empty string passed to encode(). Returning minimal BOS+EOS tensor.
2026-06-12 15:40:20,596 | modules.tokenizer | WARNING | Empty string passed to encode(). Returning minimal BOS+EOS tensor.


CELL 8: Edge Cases

── 8.1: Empty String ────────────────────────────────────
  Real tokens: 2  (BOS+EOS=2, or BOS+empty+EOS=3)
  IDs: [1, 2, 0, 0, 0]...
  ✓ Empty string handled correctly

── 8.2: Single Character 'x' ────────────────────────────
  Real tokens: 3
  IDs: [1, 212, 2, 0, 0, 0]
  ✓ Single character handled

── 8.3: String Longer Than MAX_SEQ_LEN ─────────────────
  Built overflow text:
    Words:          260
    Raw tokens:     263  (before truncation)
    MAX_SEQ_LEN:    256
    Overflow by:    7 tokens
  After encode(): 256 real tokens ✓
  EOS at position 255: 2 ✓
  PAD tokens: 0 ✓
  mask.sum(): 256 ✓
  ✓ Long string truncated correctly

── 8.4: Mathematical Symbols ────────────────────────────
  ✓ sqrt notation: '√16 = 4' → 6 tokens
  ✓ multiplication sign: '4 × 3 = 12' → 7 tokens
  ✓ division sign: '12 ÷ 4 = 3' → 7 tokens
  ✓ pi: 'π ≈ 3.14159' → 8 tokens
  ✓ not equal: 'x ≠ 0' → 5 tokens
  ✓ less/greater or eq: 'x ≤ 5 and y ≥ 2' → 9 tokens
  ✓ superscript 2: 'x² + y²

In [11]:
# ============================================================
# CELL 9: Vocabulary Coverage Analysis
#
# PURPOSE: Measure what fraction of GSM8K test tokens
#          our reduced vocabulary covers.
#
# WHY THIS MATTERS:
#   Every token NOT in our vocabulary becomes UNK.
#   The model sees UNK as a single token regardless of
#   what word it represents. High UNK rates mean:
#   - Information loss in training examples
#   - Model cannot distinguish rare-but-important words
#   - Validation metrics are artificially inflated
#     (all UNK tokens "match")
#
# THRESHOLD: 99% coverage
#   This is standard for BPE tokenizers on their target domain.
#   Below 99% means we're losing meaningful content.
#
# WHAT compute_coverage() MUST DO:
#   For each token in the test texts:
#     1. Run through BLOOM tokenizer to get bloom token IDs
#     2. Check if each bloom ID is in bloom_to_reduced
#     3. Coverage = (tokens found in mapping) / (total tokens)
# ============================================================

print("=" * 60)
print("CELL 9: Vocabulary Coverage Analysis")
print("=" * 60)

if not USING_REAL_DATA:
    print("\nSkipping full coverage test (no real GSM8K data).")
    print("Running synthetic coverage test instead...")

    # We can still test the compute_coverage() function itself
    # with our synthetic corpus

    # Test that compute_coverage() returns a float in [0, 1]
    synthetic_test = [
        "If 3x + 9 = 21, find x",
        "The answer is 4",
        "Step 1: Subtract 9",
        "Total cost: $4.25",
    ]

    coverage_synthetic = tok.compute_coverage(synthetic_test)
    print(f"\nSynthetic corpus coverage: {coverage_synthetic:.4%}")
    assert isinstance(coverage_synthetic, float), (
        f"compute_coverage() must return float, got {type(coverage_synthetic)}"
    )
    assert 0.0 <= coverage_synthetic <= 1.0, (
        f"Coverage must be in [0, 1], got {coverage_synthetic}"
    )
    print(f"  ✓ Returns float in [0, 1]")

    # Test with empty list
    coverage_empty = tok.compute_coverage([])
    print(f"\nEmpty corpus coverage: {coverage_empty}")
    assert coverage_empty == 1.0 or coverage_empty == 0.0, (
        f"Coverage of empty corpus should be 0.0 or 1.0, got {coverage_empty}"
    )
    print(f"  ✓ Empty corpus handled gracefully")

    # Test with single token text
    coverage_single = tok.compute_coverage(["x"])
    print(f"\nSingle-token coverage ('x'): {coverage_single:.4%}")
    assert 0.0 <= coverage_single <= 1.0
    print(f"  ✓ Single token handled")

    print("\n⚠ Full GSM8K coverage test skipped.")
    print("  To run: set USING_REAL_DATA=True (requires HuggingFace)")
    coverage = coverage_synthetic  # For final summary

else:
    # ── Full GSM8K Coverage Analysis ─────────────────────────
    print("\nLoading GSM8K test split for coverage analysis...")
    dataset = CELL2_RESULTS["dataset"]

    test_texts = []
    for example in dataset["test"]:
        question = example["question"].strip()
        answer = example["answer"].strip()
        if question:
            test_texts.append(question)
        if answer:
            test_texts.append(answer)

    print(f"  Test texts: {len(test_texts):,}")
    print(f"  (Questions + answers from GSM8K test split)")

    # ── Overall Coverage ──────────────────────────────────────
    print("\nComputing overall coverage...")
    coverage_start = time.perf_counter()
    coverage = tok.compute_coverage(test_texts)
    coverage_elapsed = time.perf_counter() - coverage_start

    print(f"\nOverall coverage: {coverage:.4%}")
    print(f"Computation time: {coverage_elapsed:.2f}s")

    # The main assertion
    assert coverage > 0.99, (
        f"Coverage {coverage:.4%} is below 99% threshold!\n"
        f"  Missing ~{(1 - coverage) * 100:.2f}% of test tokens.\n"
        f"  These become UNK tokens during training.\n"
        f"  Consider:\n"
        f"    1. Increasing VOCAB_SIZE in config.py\n"
        f"    2. Using a larger training corpus for vocab building\n"
        f"    3. Checking that all GSM8K training data was used"
    )
    print(f"  ✓ Coverage {coverage:.4%} > 99% threshold")

    # ── Per-Type Coverage ─────────────────────────────────────
    print("\n── Per-Type Coverage ────────────────────────────────")

    q_texts = [ex["question"] for ex in dataset["test"] if ex["question"]]
    a_texts = [ex["answer"] for ex in dataset["test"] if ex["answer"]]

    q_coverage = tok.compute_coverage(q_texts)
    a_coverage = tok.compute_coverage(a_texts)

    print(f"  Question coverage: {q_coverage:.4%}")
    print(f"  Answer coverage:   {a_coverage:.4%}")

    # Answers often have different vocabulary (arithmetic steps)
    # Both should be above threshold
    assert q_coverage > 0.99, f"Question coverage too low: {q_coverage:.4%}"
    assert a_coverage > 0.99, f"Answer coverage too low: {a_coverage:.4%}"
    print(f"  ✓ Both question and answer coverage > 99%")

    # ── Coverage Stability ────────────────────────────────────
    # Coverage should be consistent across random subsets
    print("\n── Coverage Stability Across Subsets ────────────────")
    import random
    random.seed(42)

    subset_coverages = []
    for trial in range(5):
        subset = random.sample(test_texts, min(100, len(test_texts)))
        subset_cov = tok.compute_coverage(subset)
        subset_coverages.append(subset_cov)
        print(f"  Trial {trial+1}: {subset_cov:.4%}")

    mean_cov = sum(subset_coverages) / len(subset_coverages)
    std_cov = (
        sum((c - mean_cov)**2 for c in subset_coverages) / len(subset_coverages)
    ) ** 0.5

    print(f"\n  Mean coverage:   {mean_cov:.4%}")
    print(f"  Std coverage:    {std_cov:.4%}")
    print(f"  Max deviation:   {max(abs(c - mean_cov) for c in subset_coverages):.4%}")

    assert std_cov < 0.02, (
        f"Coverage is unstable across subsets (std={std_cov:.4%}).\n"
        f"This suggests the vocabulary doesn't consistently cover the domain."
    )
    print(f"  ✓ Coverage is stable (std < 2%)")

    # ── Find Uncovered Token Examples ─────────────────────────
    # Diagnostic: what tokens ARE being missed?
    print("\n── Uncovered Token Examples (Diagnostic) ───────────")

    # We need to look at the raw BLOOM tokenization
    if hasattr(tok, 'bloom_tokenizer') and tok.bloom_tokenizer is not None:
        uncovered_tokens = []
        sample_for_analysis = test_texts[:50]

        for text in sample_for_analysis:
            bloom_ids = tok.bloom_tokenizer.encode(text, add_special_tokens=False)
            for bid in bloom_ids:
                if bid not in tok.bloom_to_reduced:
                    token_str = tok.bloom_tokenizer.decode([bid])
                    uncovered_tokens.append(token_str)

        if uncovered_tokens:
            from collections import Counter
            counts = Counter(uncovered_tokens).most_common(10)
            print(f"  Most common uncovered tokens (sample of 50 texts):")
            for token, count in counts:
                print(f"    '{token}': {count} occurrences")
        else:
            print(f"  No uncovered tokens found in sample of 50 texts")
    else:
        print(f"  (bloom_tokenizer not directly accessible for this analysis)")

print("\n" + "=" * 60)
print("✅ CELL 9 PASSED: Coverage analysis complete")
if USING_REAL_DATA:
    print(f"   GSM8K test coverage: {coverage:.2%}")
else:
    print(f"   Synthetic coverage: {coverage:.2%}")
    print(f"   (Run with real GSM8K data for full validation)")
print("=" * 60)

2026-06-12 15:40:23,865 | modules.tokenizer | INFO | Coverage: 100.0000% (24/24 tokens in vocab, 0 mapped to <unk>)
2026-06-12 15:40:23,865 | modules.tokenizer | WARNING | No tokens found in provided texts.
2026-06-12 15:40:23,866 | modules.tokenizer | INFO | Coverage: 100.0000% (1/1 tokens in vocab, 0 mapped to <unk>)


CELL 9: Vocabulary Coverage Analysis

Skipping full coverage test (no real GSM8K data).
Running synthetic coverage test instead...

Synthetic corpus coverage: 100.0000%
  ✓ Returns float in [0, 1]

Empty corpus coverage: 0.0
  ✓ Empty corpus handled gracefully

Single-token coverage ('x'): 100.0000%
  ✓ Single token handled

⚠ Full GSM8K coverage test skipped.
  To run: set USING_REAL_DATA=True (requires HuggingFace)

✅ CELL 9 PASSED: Coverage analysis complete
   Synthetic coverage: 100.00%
   (Run with real GSM8K data for full validation)


In [12]:
# ============================================================
# CELL 10: Determinism and RoPE Compatibility
#
# PURPOSE: Verify the tokenizer is position-agnostic and
#          deterministic. RoPE position encodings are added
#          in Module 2, not here.
#
# THE JEPA CONTEXT:
#   In our architecture, we encode:
#     - The question (context)     → z_context
#     - Multiple solution steps    → z_target_1, z_target_2, ...
#
#   The SAME step text might appear at different positions
#   in different training examples. The tokenizer must
#   produce identical token IDs regardless of "position."
#   Position information is baked in by RoPE in Module 2.
#
# TESTS:
#   10.1: Same text → same output (repeated calls)
#   10.2: State isolation (tok internal state doesn't drift)
#   10.3: Concurrent encoding (if using threads)
#   10.4: After batch encoding, individual encoding still works
#   10.5: After very long text, short text encodes correctly
# ============================================================

print("=" * 60)
print("CELL 10: Determinism and RoPE Compatibility")
print("=" * 60)

# ── Test 10.1: Same Text → Same Output (Many Repetitions) ─────
print("\n── Test 10.1: Repeated Encoding of Same Text ───────────")

test_texts_determinism = [
    "Subtract 9 from both sides",       # target step (short)
    "If 3x + 9 = 21, find x",          # full question
    "x = 4",                             # answer (very short)
    "Step 1: Subtract 9 from both sides. 3x = 12. Step 2: Divide. x = 4.",  # long
]

N_REPETITIONS = 20  # encode each text this many times

for text in test_texts_determinism:
    encodings = [tok.encode(text) for _ in range(N_REPETITIONS)]

    # All encodings must be identical to the first
    reference = encodings[0]
    all_identical = all(
        torch.equal(enc["input_ids"], reference["input_ids"]) and
        torch.equal(enc["attention_mask"], reference["attention_mask"])
        for enc in encodings[1:]
    )

    n_real = reference["attention_mask"].sum().item()
    print(f"  '{text[:45]}...' " if len(text) > 45 else f"  '{text}'")
    print(f"    {N_REPETITIONS} encodings identical: {all_identical}  |  {n_real} real tokens")

    assert all_identical, (
        f"Non-deterministic encoding detected!\n"
        f"  Text: '{text}'\n"
        f"  The tokenizer produced different results across {N_REPETITIONS} calls.\n"
        f"  Check for:\n"
        f"    - Random sampling in vocabulary mapping\n"
        f"    - Mutable default arguments\n"
        f"    - Global state that changes"
    )
print(f"  ✓ All {N_REPETITIONS} × {len(test_texts_determinism)} = "
      f"{N_REPETITIONS * len(test_texts_determinism)} encodings deterministic")

# ── Test 10.2: State Isolation ────────────────────────────────
# After encoding many different texts, encoding the reference
# text should still produce the same output.
print("\n── Test 10.2: State Isolation ───────────────────────────")

reference_text = "Subtract 9 from both sides"
reference_enc = tok.encode(reference_text)

# Encode lots of different texts
noise_texts = [
    f"This is text number {i} with value {i * 3.14:.2f}"
    for i in range(50)
]
for noise in noise_texts:
    _ = tok.encode(noise)

# Also do some batch encodings
_ = tok.batch_encode(noise_texts[:10])

# Reference text should still encode identically
post_noise_enc = tok.encode(reference_text)

ids_match = torch.equal(reference_enc["input_ids"], post_noise_enc["input_ids"])
mask_match = torch.equal(reference_enc["attention_mask"], post_noise_enc["attention_mask"])

print(f"  Reference text encoded before and after 50 noise encodings:")
print(f"  IDs match:  {ids_match}")
print(f"  Mask match: {mask_match}")

assert ids_match and mask_match, (
    "Tokenizer state mutated after encoding other texts!\n"
    "This suggests mutable state in the tokenizer.\n"
    "Check for caches or counters that affect output."
)
print(f"  ✓ State isolation verified")

# ── Test 10.3: Batch Does Not Affect Individual ───────────────
print("\n── Test 10.3: Batch Encoding Doesn't Affect Individual ──")

target_text = "If 3x + 9 = 21, find x"
enc_before_batch = tok.encode(target_text)

# Do a large batch encoding
large_batch = [f"Equation {i}: {i}x + {i} = {i*3}" for i in range(32)]
_ = tok.batch_encode(large_batch)

enc_after_batch = tok.encode(target_text)

assert torch.equal(enc_before_batch["input_ids"], enc_after_batch["input_ids"]), (
    "Individual encoding changed after batch_encode() was called!\n"
    "Batch encoding should not affect internal state."
)
print(f"  ✓ Individual encoding unchanged after batch_encode()")

# ── Test 10.4: Long Text Doesn't Corrupt Short Text ───────────
print("\n── Test 10.4: Long Text Doesn't Corrupt Short Text ─────")

short_text = "x = 4"
long_text_enc = "solve for x " * 50  # very long

enc_short_before = tok.encode(short_text)
_ = tok.encode(long_text_enc)  # encode long text
enc_short_after = tok.encode(short_text)

assert torch.equal(enc_short_before["input_ids"], enc_short_after["input_ids"]), (
    "Short text encoding changed after encoding a very long text!\n"
    "Possible buffer overflow or state corruption."
)
print(f"  ✓ Short text encoding unchanged after long text encoding")

# ── Test 10.5: Position Agnosticism ──────────────────────────
# Conceptual: the tokenizer does NOT add position embeddings.
# The same text encoded "as position 0" or "as position 5"
# (whatever that means) produces the same token IDs.
# Position is added later by RoPE in the attention module.
print("\n── Test 10.5: RoPE Position Encoding Note ───────────────")

step_text = "Divide both sides by 3"
enc1 = tok.encode(step_text)
enc2 = tok.encode(step_text)

assert torch.equal(enc1["input_ids"], enc2["input_ids"]), (
    "Tokenizer added position-dependent encoding. "
    "This should be done in Module 2 (embeddings), not here."
)
print(f"  ✓ Tokenizer output is position-agnostic")
print(f"  ✓ RoPE will be applied in Module 2 (embeddings)")
print(f"  Token IDs for '{step_text}':")
n_real = enc1["attention_mask"].sum().item()
print(f"    {enc1['input_ids'][:n_real].tolist()}")
print(f"    (These are the same regardless of which position the step appears at)")

# ── Store Results ─────────────────────────────────────────────
CELL10_RESULTS = {
    "deterministic": True,
    "state_isolated": True,
    "position_agnostic": True,
}

print("\n" + "=" * 60)
print("✅ CELL 10 PASSED: Tokenizer is deterministic and stateless")
print("   Same text always produces same output.")
print("   RoPE position encoding deferred to Module 2.")
print("=" * 60)

CELL 10: Determinism and RoPE Compatibility

── Test 10.1: Repeated Encoding of Same Text ───────────
  'Subtract 9 from both sides'
    20 encodings identical: True  |  8 real tokens
  'If 3x + 9 = 21, find x'
    20 encodings identical: True  |  11 real tokens
  'x = 4'
    20 encodings identical: True  |  5 real tokens
  'Step 1: Subtract 9 from both sides. 3x = 12. ...' 
    20 encodings identical: True  |  24 real tokens
  ✓ All 20 × 4 = 80 encodings deterministic

── Test 10.2: State Isolation ───────────────────────────
  Reference text encoded before and after 50 noise encodings:
  IDs match:  True
  Mask match: True
  ✓ State isolation verified

── Test 10.3: Batch Encoding Doesn't Affect Individual ──
  ✓ Individual encoding unchanged after batch_encode()

── Test 10.4: Long Text Doesn't Corrupt Short Text ─────
  ✓ Short text encoding unchanged after long text encoding

── Test 10.5: RoPE Position Encoding Note ───────────────
  ✓ Tokenizer output is position-agnostic
  ✓ Ro

In [13]:
# ============================================================
# CELL 11: Memory and Speed Benchmark
#
# PURPOSE: Verify the tokenizer is fast enough for training
#          and doesn't have memory leaks.
#
# CONTEXT:
#   Training setup:
#     - Batch size:       16 text pairs
#     - Steps per epoch:  ~450 (7,473 GSM8K examples / 16)
#     - Epochs:           100+
#   Each step calls batch_encode() twice (question + answer).
#   Tokenizer time must be negligible vs. forward/backward pass.
#
# BENCHMARKS:
#   11.1: Single encode() latency
#   11.2: batch_encode() latency at batch_size=16
#   11.3: batch_encode() scaling (sizes 1, 4, 8, 16, 32, 64)
#   11.4: Memory leak check (encode 1000 times, memory stable)
#   11.5: Throughput (texts per second)
# ============================================================

print("=" * 60)
print("CELL 11: Memory and Speed Benchmark")
print("=" * 60)

import time
import gc

# ── Helper: Stable Timing ─────────────────────────────────────
def time_function(func, n_warmup: int = 3, n_trials: int = 20) -> Tuple[float, float]:
    """
    Time a function call, returning (mean_ms, std_ms).
    Warmup calls allow JIT/caches to settle.
    """
    # Warmup
    for _ in range(n_warmup):
        func()

    # Timed trials
    times = []
    for _ in range(n_trials):
        start = time.perf_counter()
        func()
        times.append((time.perf_counter() - start) * 1000)

    mean_ms = sum(times) / len(times)
    variance = sum((t - mean_ms)**2 for t in times) / len(times)
    std_ms = variance ** 0.5

    return mean_ms, std_ms

# ── Test 11.1: Single encode() Latency ───────────────────────
print("\n── Test 11.1: Single encode() Latency ───────────────────")

single_text = "If 3x + 9 = 21, find x"
mean_single, std_single = time_function(
    lambda: tok.encode(single_text),
    n_warmup=5,
    n_trials=50,
)

print(f"  Text: '{single_text}'")
print(f"  Mean latency: {mean_single:.2f}ms ± {std_single:.2f}ms")
print(f"  Throughput:   {1000/mean_single:.0f} texts/second")

assert mean_single < 100, (
    f"Single encode() is too slow: {mean_single:.1f}ms\n"
    f"Threshold: 100ms. Check for unnecessary BLOOM model calls."
)
print(f"  ✓ Single encode < 100ms threshold")

# ── Test 11.2: Batch encode() at batch_size=16 ───────────────
print("\n── Test 11.2: batch_encode() at batch_size=16 ───────────")

batch_16_texts = [
    f"Solve problem {i}: if {i}x + {i*2} = {i*5}, find x"
    for i in range(1, 17)
]

mean_b16, std_b16 = time_function(
    lambda: tok.batch_encode(batch_16_texts),
    n_warmup=3,
    n_trials=20,
)

print(f"  Batch size: 16")
print(f"  Mean latency: {mean_b16:.1f}ms ± {std_b16:.1f}ms")
print(f"  Per-text:     {mean_b16/16:.2f}ms")
print(f"  Throughput:   {16 * 1000 / mean_b16:.0f} texts/second")

assert mean_b16 < 2000, (
    f"batch_encode(16) is too slow: {mean_b16:.1f}ms\n"
    f"Threshold: 2000ms. At this speed, tokenization will bottleneck training."
)
print(f"  ✓ batch_encode(16) < 2000ms threshold")

# ── Test 11.3: Scaling Across Batch Sizes ─────────────────────
print("\n── Test 11.3: Scaling Across Batch Sizes ────────────────")

batch_sizes = [1, 4, 8, 16, 32, 64]
scaling_results = []

print(f"  {'Batch size':>10} | {'Mean (ms)':>9} | {'Std (ms)':>8} | "
      f"{'Per-text (ms)':>13} | {'texts/sec':>9}")
print(f"  {'-'*10} | {'-'*9} | {'-'*8} | {'-'*13} | {'-'*9}")

for bs in batch_sizes:
    texts = [
        f"Solve: {i}x + {i} = {i*3}"
        for i in range(1, bs + 1)
    ]
    mean_ms, std_ms = time_function(
        lambda t=texts: tok.batch_encode(t),
        n_warmup=2,
        n_trials=10,
    )

    per_text = mean_ms / bs
    throughput = bs * 1000 / mean_ms
    scaling_results.append((bs, mean_ms, std_ms, per_text, throughput))

    print(f"  {bs:>10} | {mean_ms:>9.1f} | {std_ms:>8.1f} | "
          f"{per_text:>13.2f} | {throughput:>9.0f}")

# Verify sub-linear scaling (batching should be more efficient per text)
if len(scaling_results) >= 2:
    per_text_bs1 = scaling_results[0][3]   # per-text time at batch_size=1
    per_text_bs64 = scaling_results[-1][3]  # per-text time at batch_size=64

    print(f"\n  Per-text efficiency gain (bs=64 vs bs=1):")
    if per_text_bs64 < per_text_bs1:
        ratio = per_text_bs1 / per_text_bs64
        print(f"  ✓ {ratio:.1f}x more efficient per text at batch_size=64")
    else:
        print(f"  ⚠ No batching efficiency gain. Consider optimizing batch_encode().")

# ── Test 11.4: Memory Leak Check ─────────────────────────────
print("\n── Test 11.4: Memory Leak Check ────────────────────────")

# We can't easily measure Python memory from here without psutil,
# but we can check that many encodings don't accumulate tensors.

import gc

def check_memory_stability(n_calls: int = 100) -> bool:
    """
    Encode many times. If memory grows unboundedly,
    we have a memory leak (e.g., storing results in a list).
    """
    gc.collect()

    for i in range(n_calls):
        result = tok.encode(f"Encode call number {i}")
        # Explicitly del the result to test for ref cycles
        del result

    gc.collect()
    return True  # If we reach here without OOM, we're OK

leak_ok = check_memory_stability(200)
print(f"  Encoded 200 times without memory error: ✓")

# Also test batch encoding
for i in range(20):
    batch_texts = [f"Batch {i} item {j}" for j in range(16)]
    result = tok.batch_encode(batch_texts)
    del result

gc.collect()
print(f"  20 batches of 16 without memory error: ✓")

# ── Test 11.5: Real Training Simulation ───────────────────────
print("\n── Test 11.5: Simulated Training Loop Tokenization ─────")

# Simulate what happens per training step
# Each step: encode question batch + encode answer batch
n_steps = 50
step_texts = [
    f"Problem {i}: Find x if {i}x + {i*2} = {i*5}"
    for i in range(1, 17)
]
answer_texts = [
    f"Step 1: Subtract {i*2} from both sides. {i}x = {i*3}. x = 3."
    for i in range(1, 17)
]

start = time.perf_counter()
for step in range(n_steps):
    q_encoded = tok.batch_encode(step_texts)   # encode questions
    a_encoded = tok.batch_encode(answer_texts)  # encode answers
    del q_encoded, a_encoded
elapsed = time.perf_counter() - start

ms_per_step = (elapsed / n_steps) * 1000
print(f"  {n_steps} training steps simulated")
print(f"  Time per step (tokenization only): {ms_per_step:.1f}ms")
print(f"  (Forward pass ~100-500ms on GPU; tokenization should be << that)")

assert ms_per_step < 5000, (
    f"Tokenization too slow for training: {ms_per_step:.1f}ms per step.\n"
    f"This would add {ms_per_step * 450 / 1000 / 60:.1f} minutes per epoch."
)
print(f"  ✓ Training simulation: {ms_per_step:.0f}ms per step")

# ── Final Summary ─────────────────────────────────────────────
CELL11_RESULTS = {
    "single_encode_ms": mean_single,
    "batch16_ms": mean_b16,
    "per_text_ms": mean_b16 / 16,
    "throughput": 16 * 1000 / mean_b16,
    "training_step_ms": ms_per_step,
    "scaling_results": scaling_results,
}

print("\n" + "=" * 60)
print("✅ CELL 11 PASSED: Performance benchmarks acceptable")
print(f"   Single encode:  {mean_single:.1f}ms")
print(f"   Batch(16):      {mean_b16:.1f}ms  ({mean_b16/16:.2f}ms per text)")
print(f"   Training step:  {ms_per_step:.1f}ms  (tokenization only)")
print("=" * 60)

CELL 11: Memory and Speed Benchmark

── Test 11.1: Single encode() Latency ───────────────────
  Text: 'If 3x + 9 = 21, find x'
  Mean latency: 0.08ms ± 0.02ms
  Throughput:   12412 texts/second
  ✓ Single encode < 100ms threshold

── Test 11.2: batch_encode() at batch_size=16 ───────────
  Batch size: 16
  Mean latency: 1.5ms ± 0.3ms
  Per-text:     0.09ms
  Throughput:   10926 texts/second
  ✓ batch_encode(16) < 2000ms threshold

── Test 11.3: Scaling Across Batch Sizes ────────────────
  Batch size | Mean (ms) | Std (ms) | Per-text (ms) | texts/sec
  ---------- | --------- | -------- | ------------- | ---------
           1 |       0.1 |      0.0 |          0.08 |     12210
           4 |       0.3 |      0.1 |          0.08 |     12513
           8 |       0.6 |      0.0 |          0.07 |     13444
          16 |       1.2 |      0.0 |          0.07 |     13704
          32 |       2.2 |      0.0 |          0.07 |     14334
          64 |       4.9 |      0.4 |          0.08 |     

In [14]:
# ============================================================
# CELL 12: Final Summary and Module Handoff Report
#
# PURPOSE: Generate a complete test report. This cell is the
#          formal sign-off that Module 1 is production-ready.
#
# WHO READS THIS:
#   - You, before starting Module 2 (embeddings)
#   - Future you, debugging Module 3 (encoder)
#   - Anyone reviewing the training pipeline
#
# WHAT WE CONFIRM:
#   - All structural requirements met
#   - Performance thresholds met
#   - Edge cases handled
#   - Interface contract specified for Module 2
# ============================================================

print("=" * 60)
print("MODULE 1 FINAL TEST REPORT")
print("NanoJEPA Tokenizer — Production Readiness Assessment")
print("=" * 60)

# ── Gather All Results ────────────────────────────────────────
all_passed = True
failures = []

# Check key results from each cell
checks = [
    ("CELL 1", "Environment setup",          True),  # Would have crashed if failed
    ("CELL 2", "Data loading",               True),  # Would have crashed if failed
    ("CELL 3", "Vocabulary built",           CELL3_RESULTS["n_mapped"] > 0),
    ("CELL 3", "Special tokens correct",     True),  # Asserted in cell 3
    ("CELL 3", "No mapping collisions",      CELL3_RESULTS["n_collisions"] == 0),
    ("CELL 3", "Vocab file persists",        True),  # Asserted in cell 3
    ("CELL 4", "Output shape (256,)",        True),  # Asserted in cell 4
    ("CELL 4", "Output dtype LongTensor",    True),  # Asserted in cell 4
    ("CELL 4", "BOS at position 0",          True),  # Asserted in cell 4
    ("CELL 4", "EOS at last real position",  True),  # Asserted in cell 4
    ("CELL 4", "Padding is zero",            True),  # Asserted in cell 4
    ("CELL 4", "Attention mask correct",     True),  # Asserted in cell 4
    ("CELL 4", "All IDs in valid range",     True),  # Asserted in cell 4
    ("CELL 5", "Round-trip decode",          CELL5_RESULTS["overlap_ratio"] > 0.7),
    ("CELL 5", "Multi-text round-trip",      CELL5_RESULTS["all_round_trips_passed"]),
    ("CELL 6", "Different texts → diff IDs", CELL6_RESULTS["all_different_pairs_passed"]),
    ("CELL 6", "Deterministic encoding",     CELL6_RESULTS["determinism_verified"]),
    ("CELL 7", "Batch shape (B, 256)",       CELL7_RESULTS["consistency_verified"]),
    ("CELL 7", "Batch ≡ individual encode",  CELL7_RESULTS["consistency_verified"]),
    ("CELL 8", "Empty string handled",       True),  # Asserted in cell 8
    ("CELL 8", "Long string truncated+EOS",  True),  # Asserted in cell 8
    ("CELL 8", "Math symbols handled",       True),  # Asserted in cell 8
    ("CELL 8", "Currency/% handled",         True),  # Asserted in cell 8
    ("CELL 9", "Vocab coverage > 99%",       USING_REAL_DATA),  # Only if real data
    ("CELL 10", "Deterministic (20 calls)",  CELL10_RESULTS["deterministic"]),
    ("CELL 10", "State isolation",           CELL10_RESULTS["state_isolated"]),
    ("CELL 10", "Position-agnostic",         CELL10_RESULTS["position_agnostic"]),
    ("CELL 11", "Speed: encode < 100ms",     CELL11_RESULTS["single_encode_ms"] < 100),
    ("CELL 11", "Speed: batch(16) < 2000ms", CELL11_RESULTS["batch16_ms"] < 2000),
]

# Print results table
print(f"\n{'Cell':<8} {'Check':<40} {'Result':>7}")
print(f"{'-'*8} {'-'*40} {'-'*7}")

for cell, check, passed in checks:
    if cell == "CELL 9" and not USING_REAL_DATA:
        status = "SKIP"
        symbol = "⚠"
    elif passed:
        status = "PASS"
        symbol = "✅"
    else:
        status = "FAIL"
        symbol = "❌"
        all_passed = False
        failures.append(f"{cell}: {check}")

    print(f"{cell:<8} {check:<40} {symbol} {status}")

# ── Numeric Summary ───────────────────────────────────────────
print(f"\n── Numeric Metrics ──────────────────────────────────────")
print(f"  Vocabulary size:        {CELL3_RESULTS['n_mapped']:,} tokens mapped")
print(f"  Build time:             {CELL3_RESULTS['build_time_sec']:.2f}s")
print(f"  Load time (from disk):  {CELL3_RESULTS['load_time_sec']:.3f}s")
print(f"  Encode latency:         {CELL11_RESULTS['single_encode_ms']:.1f}ms")
print(f"  Batch(16) latency:      {CELL11_RESULTS['batch16_ms']:.1f}ms")
print(f"  Per-text latency:       {CELL11_RESULTS['per_text_ms']:.2f}ms")
print(f"  Throughput:             {CELL11_RESULTS['throughput']:.0f} texts/sec")
print(f"  Round-trip overlap:     {CELL5_RESULTS['overlap_ratio']:.1%}")
if USING_REAL_DATA:
    print(f"  GSM8K test coverage:    {coverage:.2%}")
else:
    print(f"  GSM8K test coverage:    (not measured — no real data)")

# ── Interface Contract for Module 2 ───────────────────────────
print(f"\n── Interface Contract: What Module 2 (Embeddings) Expects ─")
print(f"""
  Input to Module 2:
    result = tok.encode(text)   OR
    result = tok.batch_encode(texts)

  Tensor specifications:
    result["input_ids"]
      dtype:  torch.long (int64)
      shape:  (MAX_SEQ_LEN,)     for single encode
              (B, MAX_SEQ_LEN)   for batch encode
      values: integers in [0, VOCAB_SIZE)
              0         = PAD token
              1         = BOS token
              2         = EOS token
              [3, 32K)  = content tokens

    result["attention_mask"]
      dtype:  torch.long (int64)
      shape:  same as input_ids
      values: 1 for real tokens (including BOS and EOS)
              0 for PAD tokens

  Usage in Module 2:
    # Embedding lookup
    token_embeds = embedding_table(result["input_ids"])
    # → shape: (B, MAX_SEQ_LEN, EMBED_DIM)

    # RoPE applied HERE in Module 2, not in tokenizer
    # (tokenizer output is position-agnostic)

    # Attention uses the mask
    # mask=1 → attend, mask=0 → ignore (padding)
""")

# ── Known Limitations ─────────────────────────────────────────
print(f"── Known Limitations and Acceptable Tradeoffs ───────────")
print(f"""
  1. BPE whitespace: Decoded text may differ from input in
     whitespace around operators (+, =, ×). This is inherent
     to BLOOM's BPE vocabulary and does not affect training.

  2. Rare Unicode: Very unusual Unicode characters (emoji,
     exotic math) may map to byte fallback tokens (longer
     sequences). This is acceptable — they're rare in GSM8K.

  3. Collision-free mapping: Our 250K→32K mapping is injective
     for all tokens that appear in GSM8K. Tokens outside
     the top-32K BLOOM vocabulary are mapped to UNK or byte
     fallbacks. This affects {'<1%' if USING_REAL_DATA else 'unknown %'} of tokens.
""")

# ── Final Verdict ─────────────────────────────────────────────
print("=" * 60)
if all_passed or (not USING_REAL_DATA and len(failures) == 0):
    print("✅ MODULE 1 VERDICT: PRODUCTION READY")
    print("   All tests passed. Proceed to Module 2 (embeddings).")
else:
    print("❌ MODULE 1 VERDICT: NEEDS FIXES")
    print("   Failing checks:")
    for failure in failures:
        print(f"     - {failure}")
    print("   Fix these before implementing Module 2.")
print("=" * 60)

MODULE 1 FINAL TEST REPORT
NanoJEPA Tokenizer — Production Readiness Assessment

Cell     Check                                     Result
-------- ---------------------------------------- -------
CELL 1   Environment setup                        ✅ PASS
CELL 2   Data loading                             ✅ PASS
CELL 3   Vocabulary built                         ✅ PASS
CELL 3   Special tokens correct                   ✅ PASS
CELL 3   No mapping collisions                    ✅ PASS
CELL 3   Vocab file persists                      ✅ PASS
CELL 4   Output shape (256,)                      ✅ PASS
CELL 4   Output dtype LongTensor                  ✅ PASS
CELL 4   BOS at position 0                        ✅ PASS
CELL 4   EOS at last real position                ✅ PASS
CELL 4   Padding is zero                          ✅ PASS
CELL 4   Attention mask correct                   ✅ PASS
CELL 4   All IDs in valid range                   ✅ PASS
CELL 5   Round-trip decode                        ✅ PASS
CELL 